# 08 · Diagnosing the pre-Haiyan nighttime-lights cycle

**Question.** Is the apparent cycle in Samar–Leyte DNB-BRDF radiance associated with the source pixels, changing observability, sampling, viewing geometry, or a signal that remains unexplained?

This notebook follows one evidence chain. It first reproduces the reliability-qualified source route, then audits low-radiance days on the original grid. It fixes independent bright and dark pixels, estimates candidate periods from observed dates, checks sampling and pixel coherence, and uses STL only to describe the resulting series. Finally, it pairs measured VNP46A1 viewing angles with the **same dates and pixels** as MQF=0 VNP46A2 radiance, before testing a dark-pixel adjustment on held-out data.

**Interpretation rule.** Observability comes first. A spectral peak, STL component, or angular association is not evidence of electricity demand or service by itself. Gap-filled NTL appears only as a diagnostic comparison. The record covers 19 January 2012–7 November 2013; the 60-day NTL₀ reference is retrospective, so early-2012 patterns are descriptive.

Run the notebook from top to bottom after a kernel restart. Outputs are written under `output/baseline_cyclicity`; this notebook does not change Notebook 06 or 07.


## 1. Reuse the established source and reference route

The following cells retain Notebook 06's DNB-BRDF/MQF selection, regional P95 cap, four-day compositing, 60-day reference and fixed support. Existing profile caches are reused. These definitions are shown here so the later diagnostics can be traced back to the exact source and observability rules.


In [ ]:
from pathlib import Path
import json
import hashlib
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
from rasterio.enums import Resampling
from rasterio.features import rasterize
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (PROJECT_DIR / "datasets").exists(): PROJECT_DIR = Path('/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery')
DATA_DIR = PROJECT_DIR / "datasets"
PROCESSED_DIR = DATA_DIR / "VNP46/processed"
BASELINE_STORE = PROCESSED_DIR / "baseline/Haiyan_VNP46A2_baseline.zarr"
EXISTING_STORE = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"
GHSL_PATH = next(DATA_DIR.glob("**/GHSL_SMOD_E2015.tif"))
MUNICIPALITIES_PATH = DATA_DIR / "boundaries/MuniCities/MuniCities.shp"
OUTPUT_DIR = PROJECT_DIR / "output/baseline_characterization_v2"
FIGURE_DIR, TABLE_DIR = OUTPUT_DIR / "figures", OUTPUT_DIR / "tables"
for path in (FIGURE_DIR,TABLE_DIR): path.mkdir(parents=True,exist_ok=True)
EVENT_DATE = pd.Timestamp("2013-11-08")
BASELINE_DAYS, AGGREGATION_DAYS, KERNEL_SIZE = 60, 4, 5
BASELINE_START = EVENT_DATE-pd.Timedelta(days=BASELINE_DAYS)
PRE_EVENT_END = EVENT_DATE-pd.Timedelta(days=1)
CHARACTERIZATION_START = pd.Timestamp("2012-01-19")
CHARACTERIZATION_END = PRE_EVENT_END
GHSL_MASKS, SETTLEMENT_MASK = {"G3":(22,23,30)}, "G3"
SPATIAL_COMPLETENESS_PCT = 10.0
RQ_CLIP_PERCENTILE = 95.0
MIN_BASELINE_OBSERVATIONS = {1:5,4:3}
KNOWN_FILL_VALUES = (-9999.,-999.,-32768.,6553.5,65535.)
REBUILD_PROFILES = False  # True if source pixel values change without a date-axis change.
DNB_BAND, MQF_BAND = "DNB_BRDF_Corrected_NTL", "Mandatory_Quality_Flag"
GAP_FILLED_BAND = "Gap_Filled_DNB_BRDF_Corrected_NTL"

PLOT_FONT, PLOT_TEXT_COLOR, PLOT_GRID_COLOR = "Arial", "#243B5A", "#E8EDF3"
EVENT_COLOR, BASELINE_COLOR = "#0057FF", "#6C7882"
DNB_COLOR, GAP_FILLED_COLOR = "#0091FF", "#FF0000"
RQ_COLOR, RQ_FOUR_DAY_COLOR = "#00C54F", "#009227"
NTL_DISPLAY_MIN, NTL_DISPLAY_MAX = 0., 5.  # Display only; never truncate analytical values.
GHSL_CLASS_COLORS = {11:"#cdf57a",12:"#abcd66",13:"#375623",21:"#ffff00",22:"#a87000",23:"#732600",30:"#ff0000"}
GHSL_CLASS_LABELS = {11:"Very low density rural",12:"Low density rural",13:"Rural cluster",21:"Suburban / peri-urban",22:"Semi-dense urban",23:"Dense urban",30:"Urban centre"}
LOCATION_COLORS = {"Tacloban":"#0072B2","Palo":"#E69F00","Guiuan":"#009E73","Alangalang":"#CC79A7","Ormoc":"#D55E00","Baybay":"#56B4E9"}
SC_GREEN_COLORSCALE = [[0.,"rgba(255,255,255,0)"],[.2,"rgba(220,242,215,.25)"],
    [.4,"rgba(166,219,160,.45)"],[.6,"rgba(90,174,110,.65)"],[.8,"rgba(25,130,70,.82)"],[1.,"rgba(0,88,43,.98)"]]

def style_figure(fig,title,width=1280,height=720):
    fig.update_layout(template="plotly_white",paper_bgcolor="rgba(0,0,0,0)",plot_bgcolor="white",
        width=width,height=height,font=dict(family=PLOT_FONT,size=14,color=PLOT_TEXT_COLOR),
        title=dict(text=title,x=.03,xanchor="left",font=dict(size=22)),
        margin=dict(l=85,r=85,t=95,b=130),
        legend=dict(orientation="h",x=0,y=-.18,xanchor="left",yanchor="top"))
    fig.update_xaxes(zeroline=False,gridcolor=PLOT_GRID_COLOR)
    fig.update_yaxes(zeroline=False,gridcolor=PLOT_GRID_COLOR)
    return fig

def finish_figure(fig,filename):
    fig.write_html(FIGURE_DIR/f"{filename}.html",include_plotlyjs="directory")
    fig.show()

def add_haiyan_marker(fig,row=None,col=None):
    fig.add_vline(x=EVENT_DATE.to_pydatetime(),line_dash="dash",line_color=EVENT_COLOR,line_width=2,row=row,col=col)


In [ ]:
early = xr.open_zarr(BASELINE_STORE,chunks={},consolidated=False,mask_and_scale=False)
existing = xr.open_zarr(EXISTING_STORE,chunks={},consolidated=False,mask_and_scale=False)
if not np.all(early.processed.values == 1): raise RuntimeError("Finish Notebook 05 first.")
for axis in ("x","y"):
    np.testing.assert_allclose(early[axis].values,existing[axis].values,rtol=0,atol=1e-7)
early=early.assign_coords(x=existing.x,y=existing.y)
early["spatial_ref"]=existing["spatial_ref"]
complete=existing.processed.values==1
if len(pd.DatetimeIndex(existing.date.values[~complete]).difference(early.date.values)):
    raise RuntimeError("Existing event-period processing is unfinished.")
existing=existing.isel(date=np.flatnonzero(complete))
new_dates=pd.DatetimeIndex(early.date.values).difference(existing.date.values)
a2=xr.concat([early.sel(date=new_dates),existing],dim="date",data_vars="minimal",coords="minimal",compat="equals",join="exact").sortby("date")
a2=a2.rio.write_crs(existing.attrs["crs_wkt"])
template=a2[DNB_BAND].isel(date=0,drop=True)
ghsl=rxr.open_rasterio(GHSL_PATH,masked=True).squeeze("band",drop=True)
ghsl=ghsl.rio.reproject_match(template,resampling=Resampling.nearest).values
municipalities=gpd.read_file(MUNICIPALITIES_PATH)
region=municipalities[municipalities.PROVINCE.str.strip().str.title().isin(
    ["Samar","Eastern Samar","Northern Samar","Leyte","Southern Leyte"])].to_crs(a2.rio.crs).copy()
region["unit_name"]=region.NAME_2.str.replace("City of ","",regex=False).str.replace(" City","",regex=False).str.strip()
region["zone_id"]=np.arange(1,len(region)+1)
zone=rasterize([(g,int(i)) for g,i in zip(region.geometry,region.zone_id)],out_shape=ghsl.shape,
               transform=template.rio.transform(recalc=True),fill=0)
land=(zone>0)&np.isin(ghsl,list(GHSL_CLASS_LABELS))
land_pixels=np.flatnonzero(land)
ghsl_vector=ghsl.ravel()[land_pixels]
g3_vector=np.isin(ghsl_vector,GHSL_MASKS[SETTLEMENT_MASK])

poi_table=pd.DataFrame([
    ("Tacloban",125.0015,11.2434),("Palo",124.9904,11.1577),("Guiuan",125.7232,11.0312),
    ("Alangalang",124.8465,11.2071),("Ormoc",124.6068,11.0088),("Baybay",124.7989,10.6766)
],columns=["unit_name","longitude","latitude"])
landmarks=pd.DataFrame([
    ("Airport vicinity",125.0278,11.2275),("Port vicinity",125.0060,11.2500),
    ("Main-road vicinity",125.0030,11.2420),("Residential vicinity",125.0010,11.2650),
    ("Plaza vicinity",125.0050,11.2450),("Astrodome vicinity",125.0010,11.2240)
],columns=["unit_name","longitude","latitude"])
profiles={"Regional G3":np.flatnonzero(g3_vector)}
meta=[dict(profile="Regional G3",kind="region",class_code="G3",extent="region",color=RQ_FOUR_DAY_COLOR)]
for value,label in GHSL_CLASS_LABELS.items():
    name=f"{value} · {label}"; profiles[name]=np.flatnonzero(ghsl_vector==value)
    meta.append(dict(profile=name,kind="GHSL",class_code=str(value),extent="region",color=GHSL_CLASS_COLORS[value]))
for table,kind in [(poi_table,"centre"),(landmarks,"landmark")]:
    for point in table.itertuples():
        ix=int(abs(a2.x.values-point.longitude).argmin()); iy=int(abs(a2.y.values-point.latitude).argmin())
        for size in ([KERNEL_SIZE] if kind=="centre" else [1,3,5]):
            local=np.zeros(land.shape,dtype=bool); r=size//2
            local[max(0,iy-r):iy+r+1,max(0,ix-r):ix+r+1]=True
            if kind=="centre":
                parent_ids=region.loc[region.unit_name.eq(point.unit_name),"zone_id"].values
                local &= np.isin(zone,parent_ids)
            indices=np.flatnonzero(local.ravel()[land_pixels]&g3_vector)
            name=f"{point.unit_name} | {size}×{size}"
            profiles[name]=indices
            meta.append(dict(profile=name,kind=kind,class_code="G3",extent=f"{size}×{size}",
                color=LOCATION_COLORS.get(point.unit_name,"#174A7E")))
profile_meta=pd.DataFrame(meta)
print(f"A2: {a2.sizes['date']} available dates; {len(land_pixels):,} regional land pixels.")


In [ ]:
dates=pd.DatetimeIndex(a2.date.values)
day_blocks=np.asarray((dates-EVENT_DATE).days//AGGREGATION_DAYS)
blocks=np.arange(day_blocks.min(),day_blocks.max()+1)
cache_settings={"version":3,"stores":[str(BASELINE_STORE.resolve()),str(EXISTING_STORE.resolve())],
    "dates":dates.strftime("%Y-%m-%d").tolist(),"p95":RQ_CLIP_PERCENTILE,"reference":str(BASELINE_START),
    "minimum_reference_composites":MIN_BASELINE_OBSERVATIONS[4],"fill_values":KNOWN_FILL_VALUES,
    "aggregation_days":AGGREGATION_DAYS,"ghsl_mtime":GHSL_PATH.stat().st_mtime,
    "boundaries_mtime":MUNICIPALITIES_PATH.stat().st_mtime,
    "masks":{p:idx.tolist() for p,idx in profiles.items()}}
cache_key=hashlib.sha256(json.dumps(cache_settings,sort_keys=True).encode()).hexdigest()
cache_dir=OUTPUT_DIR/"cache"/cache_key; cache_dir.mkdir(parents=True,exist_ok=True)

def median(values,axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore",RuntimeWarning)
        return np.nanmedian(values,axis=axis)

def read_days(indices):
    batch=a2[[DNB_BAND,MQF_BAND,GAP_FILLED_BAND]].isel(date=indices).compute()
    raw=batch[DNB_BAND].values.reshape(len(indices),-1)[:,land_pixels].astype("float32")
    quality=batch[MQF_BAND].values.reshape(len(indices),-1)[:,land_pixels]
    gap=batch[GAP_FILLED_BAND].values.reshape(len(indices),-1)[:,land_pixels].astype("float32")
    raw[(quality!=0)|~np.isfinite(raw)|(raw<0)|np.isin(raw,KNOWN_FILL_VALUES)]=np.nan
    gap[~np.isfinite(gap)|(gap<0)|np.isin(gap,KNOWN_FILL_VALUES)]=np.nan
    capped=raw.copy()
    for j in range(len(indices)):
        good=raw[j,g3_vector]; good=good[np.isfinite(good)]
        if good.size: capped[j]=np.minimum(raw[j],np.percentile(good,RQ_CLIP_PERCENTILE))
    return raw,capped,gap

if not REBUILD_PROFILES and (cache_dir/"complete.json").exists():
    daily=pd.read_csv(cache_dir/"daily.csv",parse_dates=["date"])
    four_day=pd.read_csv(cache_dir/"four_day.csv",parse_dates=["date"])
    ntl0=np.load(cache_dir/"reference.npy"); reference_count=np.load(cache_dir/"reference_count.npy")
else:
    reference_cache={}; reference_composites=[]
    for block in range(-BASELINE_DAYS//AGGREGATION_DAYS,0):
        indices=np.flatnonzero(day_blocks==block)
        if len(indices):
            arrays=read_days(indices); reference_cache[block]=arrays
            reference_composites.append(median(arrays[1],axis=0))
        else: reference_composites.append(np.full(len(land_pixels),np.nan,dtype="float32"))
    reference_composites=np.asarray(reference_composites)
    reference_count=np.isfinite(reference_composites).sum(axis=0)
    ntl0=median(reference_composites,axis=0)
    reference_ok=(reference_count>=MIN_BASELINE_OBSERVATIONS[4])&np.isfinite(ntl0)&(ntl0>0)
    fixed={p:idx[reference_ok[idx]] for p,idx in profiles.items()}
    def summarize(arrays,date):
        rows=[]
        for name,idx in fixed.items():
            fresh,capped,gap=[a[idx] for a in arrays]
            valid=np.isfinite(capped); n=int(valid.sum()); sc=100*n/len(idx) if len(idx) else np.nan
            ratio=100*capped[valid].sum()/ntl0[idx][valid].sum() if n else np.nan
            rows.append((date,name,len(idx),n,sc,median(fresh),median(capped),median(gap),ratio))
        return rows
    daily_rows=[]; block_rows=[]
    for block in tqdm(blocks,desc="Observed daily and 4-day profiles"):
        indices=np.flatnonzero(day_blocks==block)
        if len(indices):
            arrays=reference_cache[block] if block in reference_cache else read_days(indices)
            for j,index in enumerate(indices): daily_rows.extend(summarize([a[j] for a in arrays],dates[index]))
            composite=[median(a,axis=0) for a in arrays]
        else: composite=[np.full(len(land_pixels),np.nan)]*3
        midpoint=EVENT_DATE+pd.Timedelta(days=int(block)*4+1.5)
        block_rows.extend(summarize(composite,midpoint))
    columns=["date","profile","fixed_pixels","valid_pixels","spatial_coverage_pct","raw_uncapped","rq_median","gap_median","reference_pct"]
    daily=pd.DataFrame(daily_rows,columns=columns); four_day=pd.DataFrame(block_rows,columns=columns)
    daily.to_csv(cache_dir/"daily.csv",index=False); four_day.to_csv(cache_dir/"four_day.csv",index=False)
    np.save(cache_dir/"reference.npy",ntl0); np.save(cache_dir/"reference_count.npy",reference_count)
    (cache_dir/"complete.json").write_text(json.dumps({"key":cache_key}))
reference_ok=(reference_count>=MIN_BASELINE_OBSERVATIONS[4])&np.isfinite(ntl0)&(ntl0>0)
fixed={p:idx[reference_ok[idx]] for p,idx in profiles.items()}
profile_meta["base_pixels"]=profile_meta.profile.map(lambda p:len(profiles[p]))
profile_meta["fixed_pixels"]=profile_meta.profile.map(lambda p:len(fixed[p]))
for frame,days in [(daily,1),(four_day,4)]:
    frame["aggregation_days"]=days
    frame["observed"]=frame.spatial_coverage_pct.ge(SPATIAL_COMPLETENESS_PCT)&frame.valid_pixels.gt(0)
    frame["raw_median_ntl"]=frame.rq_median.where(frame.observed)
    frame["normalized_ntl_pct"]=frame.reference_pct.where(frame.observed)
    frame["reference_start"],frame["reference_end"]=BASELINE_START,PRE_EVENT_END
    frame["sc_threshold_pct"],frame["p95"]=SPATIAL_COMPLETENESS_PCT,RQ_CLIP_PERCENTILE
    frame["date_start"]=frame.date-pd.to_timedelta((days-1)/2,unit="D")
    frame["date_end"]=frame.date_start+pd.to_timedelta(days-1,unit="D")
    frame["observation_status"]=np.where(frame.observed,"observed","not observable")
    frame["class_code"]=frame.profile.map(profile_meta.set_index("profile").class_code)
    frame["extent"]=frame.profile.map(profile_meta.set_index("profile").extent)
print("Fixed support — zero-pixel profiles remain visibly unavailable:")
display(profile_meta[["profile","base_pixels","fixed_pixels"]])


In [ ]:
NB8_DIR = PROJECT_DIR / "output/baseline_cyclicity"
FIGURE_DIR, TABLE_DIR = NB8_DIR / "figures", NB8_DIR / "tables"
for path in (FIGURE_DIR, TABLE_DIR):
    path.mkdir(parents=True, exist_ok=True)

baseline_daily = daily.loc[
    daily.profile.eq("Regional G3")
    & daily.date.between(CHARACTERIZATION_START, PRE_EVENT_END)
].sort_values("date").copy()
baseline_4d = four_day.loc[
    four_day.profile.eq("Regional G3")
    & four_day.date_start.ge(CHARACTERIZATION_START)
    & four_day.date_end.le(PRE_EVENT_END)
].sort_values("date").copy()
daily_calendar = pd.date_range(CHARACTERIZATION_START, PRE_EVENT_END, freq="D")
regional_daily_ntl = baseline_daily.set_index("date").raw_median_ntl.reindex(daily_calendar)
regional_daily_sc = baseline_daily.set_index("date").spatial_coverage_pct.reindex(daily_calendar)
assert len(baseline_4d) and "Regional G3" in fixed

print("Baseline dates:", CHARACTERIZATION_START.date(), "to", PRE_EVENT_END.date())
print("G3 fixed pixels:", len(fixed["Regional G3"]))
print("Daily observed / calendar days:", regional_daily_ntl.notna().sum(), "/", len(daily_calendar))
print("Four-day observed / calendar blocks:", baseline_4d.raw_median_ntl.notna().sum(), "/", len(baseline_4d))
print("Additional layers available in these Zarr stores:")
print([name for name in a2.data_vars if any(term in name.lower()
      for term in ("lunar", "cloud", "zenith", "angle", "snow"))])

## 2. Spatial source and the observed time series

The first map is the inherited reference context: GHSL morphology and 60-day pixelwise NTL on the original grid. The time-series figure below is new and restricted to the *pre-event* record. The green strip is spatial completeness; gray points are MQF-filtered daily DNB-BRDF medians before the regional P95 cap, and the green stepped curve is four-day reliability-qualified NTL. The line breaks where the SC gate fails. The comparison asks whether the cycle is already visible in daily qualified radiance or appears mainly after four-day compositing.

In [ ]:
# ------------------------------------------------------------
# Display grids — analytical values remain unchanged
# ------------------------------------------------------------

map_classes = [
    (10, "Water", "#B8D9EF"),
    *[
        (value, GHSL_CLASS_LABELS[value], GHSL_CLASS_COLORS[value])
        for value in GHSL_CLASS_LABELS
    ],
]

ghsl_grid = np.full(land.shape, np.nan)

for index, (value, _, _) in enumerate(map_classes):
    mask = (ghsl == value)
    if value != 10:
        mask &= land
    ghsl_grid[mask] = index

reference_grid = np.full(land.size, np.nan)
reference_grid[land_pixels[reference_ok]] = ntl0[reference_ok]
reference_grid = reference_grid.reshape(land.shape)

# Water is shown only where GHSL identifies water.
# Missing terrestrial NTL remains blank.
water_grid = np.where(ghsl == 10, 1.0, np.nan)

ghsl_scale = [
    [stop, color]
    for index, (_, _, color) in enumerate(map_classes)
    for stop in (
        index / len(map_classes),
        (index + 1) / len(map_classes),
    )
]

west, south, east, north = region.total_bounds
regional_bounds = (
    west - 0.03,
    south - 0.03,
    east + 0.03,
    north + 0.03,
)
tacloban_bounds = (124.94, 11.14, 125.10, 11.32)

# Dissolve internal municipality boundaries.
outline = region.dissolve().geometry.iloc[0].boundary
outline_x, outline_y = [], []

for line in (
    list(outline.geoms)
    if hasattr(outline, "geoms")
    else [outline]
):
    xx, yy = line.xy
    outline_x.extend([*xx, None])
    outline_y.extend([*yy, None])


# ------------------------------------------------------------
# Two large regional maps; Tacloban insets at their top right
# ------------------------------------------------------------

fig = go.Figure()
style_figure(fig, "", 1700, 1050)

panels = [
    # axis, grid, bounds, x-domain, y-domain, inset
    (1, ghsl_grid, regional_bounds, [0.015, 0.345], [0.22, 0.95], False),
    (2, reference_grid, regional_bounds, [0.515, 0.845], [0.22, 0.95], False),
    (3, ghsl_grid, tacloban_bounds, [0.305, 0.490], [0.62, 0.925], True),
    (4, reference_grid, tacloban_bounds, [0.805, 0.990], [0.62, 0.925], True),
]

for number, grid, bounds, x_domain, y_domain, inset in panels:
    is_ghsl = number in (1, 3)
    xref = "x" if number == 1 else f"x{number}"
    yref = "y" if number == 1 else f"y{number}"
    xaxis_name = "xaxis" if number == 1 else f"xaxis{number}"
    yaxis_name = "yaxis" if number == 1 else f"yaxis{number}"

    xi = np.flatnonzero(
        (a2.x.values >= bounds[0])
        & (a2.x.values <= bounds[2])
    )
    yi = np.flatnonzero(
        (a2.y.values >= bounds[1])
        & (a2.y.values <= bounds[3])
    )

    xx = a2.x.values[xi]
    yy = a2.y.values[yi]

    if not is_ghsl:
        fig.add_trace(go.Heatmap(
            x=xx,
            y=yy,
            z=water_grid[np.ix_(yi, xi)],
            xaxis=xref,
            yaxis=yref,
            colorscale=[[0, "#B8D9EF"], [1, "#B8D9EF"]],
            zmin=0,
            zmax=1,
            showscale=False,
            hoverinfo="skip",
        ))

    fig.add_trace(go.Heatmap(
        x=xx,
        y=yy,
        z=grid[np.ix_(yi, xi)],
        xaxis=xref,
        yaxis=yref,
        colorscale=ghsl_scale if is_ghsl else "Inferno",
        zmin=-0.5 if is_ghsl else NTL_DISPLAY_MIN,
        zmax=len(map_classes) - 0.5 if is_ghsl else NTL_DISPLAY_MAX,
        showscale=(number == 2),
        zsmooth=False,
        hoverongaps=False,
        hoverinfo="skip" if is_ghsl else None,
        hovertemplate=(
            None if is_ghsl else
            "%{y:.4f}°N, %{x:.4f}°E"
            "<br>NTL: %{z:.3f} nW cm⁻² sr⁻¹<extra></extra>"
        ),
        colorbar=dict(
            orientation="h",
            title=dict(
                text="DNB-BRDF nighttime lights (nW cm⁻² sr⁻¹)",
                side="top",
            ),
            x=0.735,
            xanchor="center",
            y=0.11,
            yanchor="middle",
            len=0.36,
            thickness=18,
            outlinewidth=0,
            tickvals=np.linspace(NTL_DISPLAY_MIN, NTL_DISPLAY_MAX, 6),
        ),
    ))

    # Outer outline only: fine enough to preserve raster visibility.
    fig.add_trace(go.Scatter(
        x=outline_x,
        y=outline_y,
        xaxis=xref,
        yaxis=yref,
        mode="lines",
        line=dict(color="rgba(36,59,90,0.28)", width=0.4),
        showlegend=False,
        hoverinfo="skip",
    ))

    fig.update_layout({
        xaxis_name: dict(
            domain=x_domain,
            anchor=yref,
            range=[bounds[0], bounds[2]],
            constrain="domain",
            showgrid=False,
            zeroline=False,
            showticklabels=not inset,
            ticks="" if inset else "outside",
            tickvals=np.arange(
                np.ceil(bounds[0]), np.floor(bounds[2]) + 1, 1
            ),
            tickformat=".0f",
            ticksuffix="°E",
            title=None,
        ),
        yaxis_name: dict(
            domain=y_domain,
            anchor=xref,
            range=[bounds[1], bounds[3]],
            scaleanchor=xref,
            scaleratio=1 / np.cos(np.deg2rad(
                (bounds[1] + bounds[3]) / 2
            )),
            constrain="domain",
            showgrid=False,
            zeroline=False,
            showticklabels=not inset,
            ticks="" if inset else "outside",
            tickvals=np.arange(
                np.ceil(bounds[1]), np.floor(bounds[3]) + 1, 1
            ),
            tickformat=".0f",
            ticksuffix="°N",
            title=None,
        ),
    })

    if inset:
        # Frame and labels only inside the inset.
        fig.add_shape(
            type="rect",
            xref=xref,
            yref=yref,
            x0=bounds[0],
            x1=bounds[2],
            y0=bounds[1],
            y1=bounds[3],
            line=dict(color=PLOT_TEXT_COLOR, width=1),
            fillcolor="rgba(0,0,0,0)",
        )

        visible = poi_table[
            poi_table.longitude.between(bounds[0], bounds[2])
            & poi_table.latitude.between(bounds[1], bounds[3])
        ]

        fig.add_trace(go.Scatter(
            x=visible.longitude,
            y=visible.latitude,
            text=visible.unit_name,
            xaxis=xref,
            yaxis=yref,
            mode="markers+text",
            marker=dict(
                color="white",
                size=5,
                line=dict(color=PLOT_TEXT_COLOR, width=0.8),
            ),
            textposition="top center",
            textfont=dict(size=11, color=PLOT_TEXT_COLOR),
            showlegend=False,
        ))

    else:
        # Locate the inset footprint without six overlapping regional labels.
        fig.add_shape(
            type="rect",
            xref=xref,
            yref=yref,
            x0=tacloban_bounds[0],
            x1=tacloban_bounds[2],
            y0=tacloban_bounds[1],
            y1=tacloban_bounds[3],
            line=dict(color="#64748B", width=1, dash="dot"),
            fillcolor="rgba(0,0,0,0)",
        )


# ------------------------------------------------------------
# Panel headings — no main title
# ------------------------------------------------------------

for x, text in [
    (0.18, "GHSL settlement context"),
    (0.68, "60-day pixel baseline"),
]:
    fig.add_annotation(
        x=x, y=0.995,
        xref="paper", yref="paper",
        text=f"{text}",
        showarrow=False,
        font=dict(size=19, color=PLOT_TEXT_COLOR),
    )

for x in (0.3975, 0.8975):
    fig.add_annotation(
        x=x, y=0.945,
        xref="paper", yref="paper",
        text="Tacloban",
        showarrow=False,
        font=dict(size=14, color=PLOT_TEXT_COLOR),
    )


# ------------------------------------------------------------
# Fixed GHSL legend: two columns × four rows, including water
# ------------------------------------------------------------

for index, (value, label, color) in enumerate(map_classes):
    column, row = divmod(index, 4)
    x = 0.055 + column * 0.235
    y = 0.165 - row * 0.042

    fig.add_shape(
        type="rect",
        xref="paper", yref="paper",
        x0=x, x1=x + 0.008,
        y0=y - 0.006, y1=y + 0.006,
        fillcolor=color,
        line=dict(width=0),
    )
    fig.add_annotation(
        x=x + 0.013, y=y,
        xref="paper", yref="paper",
        text=f"{value} · {label}",
        showarrow=False,
        xanchor="left",
        yanchor="middle",
        font=dict(size=17, color=PLOT_TEXT_COLOR),
    )

fig.update_layout(
    title_text="",
    showlegend=False,
    margin=dict(l=55, r=35, t=35, b=25),
)


# Same road shapefile and line treatment as Notebook 06's pixel-context map.
road_path = globals().get("ROADS_PATH")
if road_path is None:
    road_files = sorted(DATA_DIR.glob("**/*[Rr]oad*.shp"))
    road_path = road_files[0] if road_files else None
if road_path is None:
    raise FileNotFoundError(
        "Set ROADS_PATH to the road shapefile used by Notebook 06."
    )
roads_local = (
    gpd.read_file(road_path).to_crs(a2.rio.crs)
    .cx[regional_bounds[0]:regional_bounds[2],
        regional_bounds[1]:regional_bounds[3]]
)
road_x, road_y = [], []
for geometry in roads_local.geometry.dropna():
    parts = geometry.geoms if hasattr(geometry, "geoms") else [geometry]
    for part in parts:
        if hasattr(part, "xy"):
            xx, yy = part.xy
            road_x.extend([*xx, None])
            road_y.extend([*yy, None])
for number in (1, 2, 3, 4):
    xref = "x" if number == 1 else f"x{number}"
    yref = "y" if number == 1 else f"y{number}"
    is_ntl = number in (2, 4)
    fig.add_trace(go.Scatter(
        x=road_x, y=road_y, xaxis=xref, yaxis=yref,
        mode="lines",
        line=dict(
            color=("rgba(240,245,250,.55)" if is_ntl
                   else "rgba(36,59,90,.30)"),
            width=.8 if number in (3, 4) else .45,
        ),
        hoverinfo="skip", showlegend=False,
    ))
print("Road shapefile:", road_path)

finish_figure(fig, "01_context_and_reference")

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    row_heights=[.13, .87], vertical_spacing=.055)
fig.add_trace(go.Heatmap(
    x=baseline_daily.date, y=["SC"],
    z=[baseline_daily.spatial_coverage_pct.to_numpy()],
    colorscale=SC_GREEN_COLORSCALE, zmin=0, zmax=100,
    colorbar=dict(title="SC (%)", thickness=17, tickfont=dict(size=15), len=.18, y=.91),
    hovertemplate="%{x|%d %b %Y}<br>SC %{z:.1f}%<extra></extra>",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=baseline_daily.date, y=baseline_daily.raw_uncapped,
    mode="markers", marker=dict(color="#929BA5", size=4, opacity=.42),
    name="Daily DNB-BRDF · MQF 0, uncapped",
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=baseline_4d.date, y=baseline_4d.raw_median_ntl,
    mode="lines", connectgaps=False,
    line=dict(color=RQ_FOUR_DAY_COLOR, width=2.0, shape="hv"),
    name="RQ NTL (4D)",
), row=2, col=1)
style_figure(fig, "", 1550, 680)
fig.update_xaxes(range=[CHARACTERIZATION_START, PRE_EVENT_END],
    dtick="M3", tickformat="%b %Y", row=2, col=1)
fig.update_yaxes(title_text="NTL (nW cm⁻² sr⁻¹)", rangemode="tozero", row=2, col=1)
for month in pd.date_range(CHARACTERIZATION_START.to_period("M").to_timestamp(),
                           PRE_EVENT_END, freq="MS"):
    fig.add_vline(x=month, line_color="rgba(36,59,90,.10)",
                  line_width=.8, layer="below", row=2, col=1)
finish_figure(fig, "01_pre_event_observed_and_support")

## 3. Reproduce near-zero days before interpreting them

The diagnostic uses every pre-Haiyan day whose uncapped, MQF-filtered regional median is ≤0.05 nW cm⁻² sr⁻¹. First compare the same source pixels before and after MQF and capping. Then compare each day's visible-pixel NTL₀ with the NTL₀ of all fixed G3 pixels. If the visible reference is much dimmer, changing support is a plausible contributor; if normally bright reference pixels are visible but their observed radiance is near the 0.01 floor, composition alone is insufficient. The four-day check tests the pixel-then-spatial aggregation order.

The maps contrast a near-zero day that **passes** the SC gate with one that **fails** it when both exist. They use Notebook 06's exact regional domains and Tacloban inset positions, its coastline outline, the road shapefile, the same NTL color scale and the same geographic scaling. One paired figure compares source DNB-BRDF with MQF=0 on that day. The second compares the four-day composite with the reference radiance of precisely the pixels visible in that composite. White terrestrial cells are outside fixed G3 support or missing on that date; neither means observed darkness.

In [ ]:
LOW_NTL = .05
low_dates = baseline_daily.loc[
    baseline_daily.raw_uncapped.le(LOW_NTL), "date"
].drop_duplicates().sort_values()
low_audit_path = TABLE_DIR / f"near_zero_daily_audit_{cache_key[:10]}.csv"
if low_audit_path.exists() and not REBUILD_PROFILES:
    low_audit = pd.read_csv(low_audit_path, parse_dates=["date"])
else:
    g3_idx = fixed["Regional G3"]
    records = []
    requested = dates.get_indexer(pd.DatetimeIndex(low_dates))
    if (requested < 0).any():
        raise ValueError("Near-zero date is missing from the source-date axis.")
    for start in tqdm(range(0, len(requested), 12), desc="Audit near-zero days"):
        indices = requested[start:start+12]
        fresh, capped, _ = read_days(indices)
        for j, source_idx in enumerate(indices):
            values = fresh[j, g3_idx]
            valid = np.isfinite(values)
            observed_reference = ntl0[g3_idx][valid]
            row = dict(
                date=dates[source_idx], valid_pixels=int(valid.sum()),
                SC_pct=100*valid.mean(),
                daily_median_recomputed=median(values),
                capped_median_recomputed=median(capped[j, g3_idx]),
                reference_median_of_visible_pixels=median(observed_reference),
                reference_median_all_fixed=median(ntl0[g3_idx]),
                at_0_01_pct=100*np.isclose(values[valid], .01, atol=.001).mean(),
                below_0_05_pct=100*(values[valid] <= .05).mean(),
                capped_pixel_pct=(100*np.mean(capped[j, g3_idx][valid] < values[valid])
                                  if valid.any() else np.nan),
            )
            for cls in (22, 23, 30):
                in_class = valid & (ghsl_vector[g3_idx] == cls)
                row[f"class_{cls}_visible"] = int(in_class.sum())
                row[f"class_{cls}_median"] = median(values[in_class]) if in_class.any() else np.nan
            records.append(row)
    low_audit = pd.DataFrame(records)
    low_audit.to_csv(low_audit_path, index=False)

# Gap-filled NTL is a diagnostic comparator, never an observed substitute.
if "gap_median" not in low_audit.columns:
    low_audit = low_audit.merge(
        baseline_daily[["date", "gap_median"]], on="date", how="left",
        validate="one_to_one",
    )
print("Median gap-filled NTL on near-zero dates (diagnostic only):",
      round(low_audit.gap_median.median(), 3))
print(f"Near-zero MQF-filtered daily medians: {len(low_audit)}")
print(f"Passing the {SPATIAL_COMPLETENESS_PCT:g}% SC gate:",
      low_audit.SC_pct.ge(SPATIAL_COMPLETENESS_PCT).sum())
display(low_audit.sort_values("SC_pct", ascending=False).head(20).round(3))
display(low_audit.groupby(low_audit.date.dt.to_period("M")).agg(
    days=("date", "size"), median_SC_pct=("SC_pct", "median"),
    median_floor_share_pct=("at_0_01_pct", "median"),
).round(1))

# The reference of exactly the visible pixels tests a changing-contributor
# explanation without assuming that all missing pixels were dark.
fig = make_subplots(rows=1, cols=2, horizontal_spacing=.12,
    subplot_titles=["Coverage versus near-zero radiance",
                    "Reference brightness of pixels visible that day"])
fig.add_trace(go.Scatter(
    x=low_audit.SC_pct, y=low_audit.daily_median_recomputed,
    mode="markers", marker=dict(
        size=9, color=low_audit.at_0_01_pct, colorscale="Viridis",
        cmin=0, cmax=100, showscale=True,
        colorbar=dict(title="At 0.01 (%)", thickness=16)),
    text=low_audit.date.dt.strftime("%d %b %Y"),
    name="Near-zero days",
    hovertemplate="%{text}<br>SC %{x:.1f}%<br>Median %{y:.3f}<extra></extra>"),
    row=1, col=1)
fig.add_trace(go.Scatter(
    x=low_audit.reference_median_of_visible_pixels,
    y=low_audit.daily_median_recomputed,
    mode="markers", marker=dict(
        size=9, color=low_audit.SC_pct, colorscale=SC_GREEN_COLORSCALE,
        cmin=0, cmax=100), text=low_audit.date.dt.strftime("%d %b %Y"),
    name="Same visible pixels",
    hovertemplate=("%{text}<br>Visible-pixel NTL₀ %{x:.2f}"
                   "<br>Observed median %{y:.3f}<extra></extra>")),
    row=1, col=2)
style_figure(fig, "Near-zero regional days · composition check", 1400, 560)
fig.update_xaxes(title_text="SC (%)", row=1, col=1)
fig.update_xaxes(title_text="NTL₀ of observed pixels (nW cm⁻² sr⁻¹)",
                 row=1, col=2)
fig.update_yaxes(title_text="Daily DNB-BRDF median (nW cm⁻² sr⁻¹)",
                 row=1, col=1)
finish_figure(fig, "02_near_zero_composition")

In [ ]:
# Compare the pixelwise-then-spatial 4D RQ calculation against a median
# of four daily regional medians. These need not be equal.
low_blocks = baseline_4d.loc[baseline_4d.raw_median_ntl.le(LOW_NTL)].copy()
print("Near-zero observed four-day blocks:", len(low_blocks))
selected_blocks = low_blocks.sort_values("spatial_coverage_pct", ascending=False).head(3)
g3_idx = fixed["Regional G3"]
block_rows = []
for block in selected_blocks.itertuples():
    source_dates = dates[(dates >= block.date_start) & (dates <= block.date_end)]
    indices = dates.get_indexer(source_dates)
    fresh, capped, _ = read_days(indices)
    pixel_4d = median(capped[:, g3_idx], axis=0)
    valid = np.isfinite(pixel_4d)
    daily_regional_medians = [median(capped[j, g3_idx]) for j in range(len(indices))]
    block_rows.append(dict(
        date=block.date, source_days=len(indices),
        cached_4D_RQ=block.rq_median,
        recomputed_pixel_then_spatial=median(pixel_4d),
        median_of_daily_regional_medians=median(daily_regional_medians),
        valid_pixels=int(valid.sum()), SC_pct=100*valid.mean(),
        at_0_01_pct=100*np.isclose(pixel_4d[valid], .01, atol=.001).mean(),
        reference_median_of_visible_pixels=median(ntl0[g3_idx][valid]),
    ))
display(pd.DataFrame(block_rows).round(3))

In [ ]:
# Reuse the road lines and exact map domains from the context figure above.
def show_near_zero_map_pair(left_values, right_values, headings, filename):
    # Domains, geographic scaling, and inset placement match Notebook 06.
    display_grids = []
    g3_idx = fixed["Regional G3"]
    for values in (left_values, right_values):
        grid = np.full(land.shape, np.nan)
        grid.ravel()[land_pixels[g3_idx]] = values[g3_idx]
        display_grids.append(grid)

    fig = go.Figure()
    style_figure(fig, "", 1700, 1050)
    panels = [
        (1, display_grids[0], regional_bounds, [0.015, 0.345], [0.22, 0.95], False),
        (2, display_grids[1], regional_bounds, [0.515, 0.845], [0.22, 0.95], False),
        (3, display_grids[0], tacloban_bounds, [0.305, 0.490], [0.62, 0.925], True),
        (4, display_grids[1], tacloban_bounds, [0.805, 0.990], [0.62, 0.925], True),
    ]
    for number, grid, bounds, x_domain, y_domain, inset in panels:
        xref = "x" if number == 1 else f"x{number}"
        yref = "y" if number == 1 else f"y{number}"
        xaxis_name = "xaxis" if number == 1 else f"xaxis{number}"
        yaxis_name = "yaxis" if number == 1 else f"yaxis{number}"
        xi = np.flatnonzero(
            (a2.x.values >= bounds[0]) & (a2.x.values <= bounds[2])
        )
        yi = np.flatnonzero(
            (a2.y.values >= bounds[1]) & (a2.y.values <= bounds[3])
        )
        xx, yy = a2.x.values[xi], a2.y.values[yi]
        fig.add_trace(go.Heatmap(
            x=xx, y=yy, z=water_grid[np.ix_(yi, xi)],
            xaxis=xref, yaxis=yref,
            colorscale=[[0, "#B8D9EF"], [1, "#B8D9EF"]],
            zmin=0, zmax=1, showscale=False, hoverinfo="skip",
        ))
        fig.add_trace(go.Heatmap(
            x=xx, y=yy, z=grid[np.ix_(yi, xi)],
            xaxis=xref, yaxis=yref, colorscale="Inferno",
            zmin=NTL_DISPLAY_MIN, zmax=NTL_DISPLAY_MAX,
            zsmooth=False, hoverongaps=False, showscale=(number == 2),
            hovertemplate=("%{y:.4f}°N, %{x:.4f}°E"
                           "<br>NTL %{z:.3f} nW cm⁻² sr⁻¹<extra></extra>"),
            colorbar=dict(
                orientation="h",
                title=dict(text="DNB-BRDF nighttime lights (nW cm⁻² sr⁻¹)",
                           side="top", font=dict(size=16)),
                x=0.735, xanchor="center", y=0.11, yanchor="middle",
                len=0.36, thickness=18, outlinewidth=0,
                tickvals=np.linspace(NTL_DISPLAY_MIN, NTL_DISPLAY_MAX, 6),
                tickfont=dict(size=15),
            ),
        ))
        fig.add_trace(go.Scatter(
            x=outline_x, y=outline_y, xaxis=xref, yaxis=yref,
            mode="lines", line=dict(color="rgba(36,59,90,.28)", width=.4),
            showlegend=False, hoverinfo="skip",
        ))
        fig.add_trace(go.Scatter(
            x=road_x, y=road_y, xaxis=xref, yaxis=yref,
            mode="lines",
            line=dict(color="rgba(240,245,250,.50)",
                      width=.8 if inset else .45),
            showlegend=False, hoverinfo="skip",
        ))
        fig.update_layout({
            xaxis_name: dict(
                domain=x_domain, anchor=yref,
                range=[bounds[0], bounds[2]], constrain="domain",
                showgrid=False, zeroline=False,
                showticklabels=not inset,
                ticks="" if inset else "outside",
                tickvals=np.arange(np.ceil(bounds[0]),
                                   np.floor(bounds[2]) + 1, 1),
                tickformat=".0f", ticksuffix="°E", title=None,
            ),
            yaxis_name: dict(
                domain=y_domain, anchor=xref,
                range=[bounds[1], bounds[3]],
                scaleanchor=xref,
                scaleratio=1 / np.cos(np.deg2rad(
                    (bounds[1] + bounds[3]) / 2)),
                constrain="domain", showgrid=False, zeroline=False,
                showticklabels=not inset,
                ticks="" if inset else "outside",
                tickvals=np.arange(np.ceil(bounds[1]),
                                   np.floor(bounds[3]) + 1, 1),
                tickformat=".0f", ticksuffix="°N", title=None,
            ),
        })
        if inset:
            fig.add_shape(
                type="rect", xref=xref, yref=yref,
                x0=bounds[0], x1=bounds[2],
                y0=bounds[1], y1=bounds[3],
                line=dict(color=PLOT_TEXT_COLOR, width=1),
                fillcolor="rgba(0,0,0,0)",
            )
            visible_points = poi_table[
                poi_table.longitude.between(bounds[0], bounds[2])
                & poi_table.latitude.between(bounds[1], bounds[3])
            ]
            fig.add_trace(go.Scatter(
                x=visible_points.longitude, y=visible_points.latitude,
                text=visible_points.unit_name,
                xaxis=xref, yaxis=yref, mode="markers+text",
                marker=dict(color="white", size=5,
                            line=dict(color=PLOT_TEXT_COLOR, width=.8)),
                textposition="top center",
                textfont=dict(size=11, color=PLOT_TEXT_COLOR),
                showlegend=False, hoverinfo="skip",
            ))
        else:
            fig.add_shape(
                type="rect", xref=xref, yref=yref,
                x0=tacloban_bounds[0], x1=tacloban_bounds[2],
                y0=tacloban_bounds[1], y1=tacloban_bounds[3],
                line=dict(color="#64748B", width=1, dash="dot"),
                fillcolor="rgba(0,0,0,0)",
            )

    for x, label in [(0.18, headings[0]), (0.68, headings[1])]:
        fig.add_annotation(
            x=x, y=.995, xref="paper", yref="paper", text=label,
            showarrow=False, font=dict(size=19, color=PLOT_TEXT_COLOR),
        )
    for x in (.3975, .8975):
        fig.add_annotation(
            x=x, y=.945, xref="paper", yref="paper", text="Tacloban",
            showarrow=False, font=dict(size=14, color=PLOT_TEXT_COLOR),
        )
    fig.update_layout(title_text="", showlegend=False,
                      margin=dict(l=55, r=35, t=35, b=25))
    finish_figure(fig, filename)

# Contrast a well-observed low day with an observation-limited low day.
eligible_low = low_audit.loc[
    low_audit.SC_pct.ge(SPATIAL_COMPLETENESS_PCT)
].sort_values("SC_pct", ascending=False)
poor_low = low_audit.loc[
    low_audit.SC_pct.lt(SPATIAL_COMPLETENESS_PCT)
].sort_values("SC_pct", ascending=False)
map_dates = []
if not eligible_low.empty:
    map_dates.append(eligible_low.date.iloc[0])
if not poor_low.empty:
    map_dates.append(poor_low.date.iloc[0])
if len(map_dates) < 2:
    remaining = low_audit.loc[~low_audit.date.isin(map_dates)].sort_values(
        "SC_pct", ascending=False
    )
    for date in remaining.date:
        if not map_dates or min(abs(date - earlier).days for earlier in map_dates) >= 60:
            map_dates.append(date)
            break
if not map_dates:
    print("No near-zero source days to map.")

for map_date in map_dates:
    source_index = int(dates.get_loc(map_date))
    block_id = int(day_blocks[source_index])
    block_indices = np.flatnonzero(day_blocks == block_id)
    source = a2[DNB_BAND].isel(date=source_index).compute().values
    source_layer = source.ravel()[land_pixels].astype(float)
    source_layer[(source_layer < 0) | ~np.isfinite(source_layer)
                 | np.isin(source_layer, KNOWN_FILL_VALUES)] = np.nan
    fresh, capped, _ = read_days(block_indices)
    day_fresh = fresh[list(block_indices).index(source_index)]
    composite = median(capped, axis=0)
    g3_idx = fixed["Regional G3"]
    reference_same = np.full(len(land_pixels), np.nan)
    visible = np.isfinite(composite[g3_idx])
    reference_same[g3_idx[visible]] = ntl0[g3_idx[visible]]

    case_sc = float(low_audit.set_index("date").loc[map_date, "SC_pct"])
    block_sc = 100*visible.mean()
    print(f"Near-zero case: {map_date:%d %b %Y}; "
          f"daily SC {case_sc:.1f}%; four-day SC {block_sc:.1f}%")
    show_near_zero_map_pair(
        source_layer, day_fresh,
        (f"Source DNB-BRDF · {map_date:%d %b %Y}",
         f"MQF = 0 · SC {case_sc:.1f}%"),
        f"02_source_mqf_{map_date:%Y%m%d}",
    )
    show_near_zero_map_pair(
        composite, reference_same,
        (f"RQ NTL (4D) · {map_date:%d %b %Y}",
         f"NTL₀ · matched {block_sc:.0f}% SC"),
        f"02_four_day_reference_{map_date:%Y%m%d}",
    )

## 4. Fix the darkest and brightest pixels before examining their histories

The controls are chosen **once** from the 60-day reference, then held fixed through the whole pre-event calendar. The low-reference group comes from rural GHSL classes 11–13 and the high-reference group from urban-centre class 30. These are deliberately extreme pixels, not representative regional averages. Where support allows it, dark donors and dark holdouts come from **different municipalities**; the cell reports when that spatial split is impossible. The map exposes any remaining geographic concentration.

The reference's proximity to the DNB-BRDF floor is a critical limitation. A dark group that spends much of its observed record at 0.01 cannot provide a useful subtractive control even if it shares a spectral peak.

In [ ]:
N_CONTROL = 40
eligible_reference = reference_ok & np.isfinite(ntl0)
dark_rank = np.flatnonzero(eligible_reference & np.isin(ghsl_vector, [11, 12, 13]))
bright_rank = np.flatnonzero(eligible_reference & (ghsl_vector == 30))
dark_rank = dark_rank[np.argsort(ntl0[dark_rank])][:N_CONTROL]
bright_rank = bright_rank[np.argsort(ntl0[bright_rank])][-N_CONTROL:]
if len(dark_rank) < 20 or len(bright_rank) < 20:
    raise ValueError("Too few eligible dark or bright reference pixels.")
# Prefer a municipality-disjoint split so validation is spatially distinct.
dark_zone = zone.ravel()[land_pixels[dark_rank]]
unique_dark_zones = np.sort(np.unique(dark_zone))
dark_donor = dark_rank[np.isin(dark_zone, unique_dark_zones[::2])]
dark_holdout = dark_rank[np.isin(dark_zone, unique_dark_zones[1::2])]
if min(len(dark_donor), len(dark_holdout)) < 8:
    dark_donor, dark_holdout = dark_rank[::2], dark_rank[1::2]
    print("Dark controls split by alternating reference rank; "
          "too few municipality-distinct pixels for a spatial split.")
else:
    print("Dark donor and holdout use disjoint municipalities.")
pixel_groups = {
    "Dark donor": dark_donor,
    "Dark holdout": dark_holdout,
    "Bright urban": bright_rank,
}
selected_pixels = np.concatenate(list(pixel_groups.values()))
selected_flat = land_pixels[selected_pixels]
selected_y, selected_x = np.unravel_index(selected_flat, land.shape)
group_id = np.concatenate([
    np.repeat(name, len(pixels)) for name, pixels in pixel_groups.items()
])
controls = pd.DataFrame({
    "group": group_id,
    "pixel_index": selected_pixels,
    "x": a2.x.values[selected_x],
    "y": a2.y.values[selected_y],
    "GHSL": ghsl_vector[selected_pixels].astype(int),
    "NTL0": ntl0[selected_pixels],
    "municipality_id": zone.ravel()[selected_flat],
})
display(controls.groupby("group").agg(
    pixels=("pixel_index", "size"),
    reference_min=("NTL0", "min"),
    reference_median=("NTL0", "median"),
    reference_max=("NTL0", "max"),
    municipalities=("municipality_id", "nunique"),
).round(3))
controls.to_csv(TABLE_DIR / "fixed_dark_bright_pixels.csv", index=False)

fig = go.Figure()
fig.add_trace(go.Heatmap(
    x=a2.x.values, y=a2.y.values, z=reference_grid,
    colorscale="Inferno", zmin=0, zmax=NTL_DISPLAY_MAX,
    colorbar=dict(title="NTL₀", thickness=18, tickfont=dict(size=15)),
    hovertemplate="NTL₀ %{z:.2f}<extra></extra>",
))
for name, color, symbol in [
    ("Dark donor", "#56B4E9", "circle-open"),
    ("Dark holdout", "#0072B2", "diamond-open"),
    ("Bright urban", "#FF5C3D", "square-open"),
]:
    group = controls.loc[controls.group.eq(name)]
    fig.add_trace(go.Scatter(
        x=group.x, y=group.y, mode="markers",
        marker=dict(symbol=symbol, size=10, color=color,
                    line=dict(color=color, width=1.5)),
        name=name, customdata=group[["NTL0", "GHSL", "municipality_id"]],
        hovertemplate=("%{x:.3f}°E, %{y:.3f}°N<br>NTL₀ %{customdata[0]:.2f}"
                       "<br>GHSL %{customdata[1]:.0f}"
                       "<br>Municipality ID %{customdata[2]:.0f}<extra></extra>"),
    ))
style_figure(fig, "Fixed low- and high-reference pixels", 1450, 800)
fig.update_xaxes(title_text="Longitude", scaleanchor="y", scaleratio=1)
fig.update_yaxes(title_text="Latitude")
finish_figure(fig, "03_fixed_dark_bright_locations")

## 5. Extract fixed-pixel daily radiance and observability

Each selected pixel contributes only its own finite, nonnegative MQF=0 DNB-BRDF value. The group median uses **all dates with at least one observed pixel**; no additional 50% spatial-completeness rule is imposed in this diagnostic. The aligned green strip and pixel counts show the varying support. Near-floor share uses valid pixels only.


In [ ]:
selected_dates = dates[(dates >= CHARACTERIZATION_START) & (dates <= PRE_EVENT_END)]
selected_indices = dates.get_indexer(selected_dates)
pixel_values = np.full((len(selected_dates), len(selected_pixels)), np.nan, dtype="float32")
pixel_fresh = np.zeros(pixel_values.shape, dtype=bool)
for start in tqdm(range(0, len(selected_indices), 32), desc="Read control pixels"):
    sl = slice(start, min(start + 32, len(selected_indices)))
    batch = a2[[DNB_BAND, MQF_BAND]].isel(
        date=selected_indices[sl],
        y=xr.DataArray(selected_y, dims="sample_pixel"),
        x=xr.DataArray(selected_x, dims="sample_pixel"),
    ).compute()
    values = batch[DNB_BAND].transpose("date", "sample_pixel").values.astype("float32")
    quality = batch[MQF_BAND].transpose("date", "sample_pixel").values
    good = ((quality == 0) & np.isfinite(values) & (values >= 0)
            & ~np.isin(values, KNOWN_FILL_VALUES))
    pixel_values[sl] = np.where(good, values, np.nan)
    pixel_fresh[sl] = good

group_daily = pd.DataFrame(index=selected_dates)
group_summary = []
offset = 0
for name, indices in pixel_groups.items():
    sub = pixel_values[:, offset:offset + len(indices)]
    visible = np.isfinite(sub)
    sc = 100 * visible.mean(axis=1)
    raw_med = median(sub, axis=1)
    group_daily[f"{name} SC"] = sc
    group_daily[f"{name} NTL"] = raw_med
    group_daily[f"{name} valid pixels"] = visible.sum(axis=1)
    group_daily[f"{name} floor %"] = np.where(
        visible.sum(axis=1) > 0,
        100 * np.isclose(sub, .01, atol=.001).sum(axis=1)
        / np.maximum(visible.sum(axis=1), 1), np.nan
    )
    group_summary.append(dict(
        group=name, fixed_pixels=len(indices),
        observed_days=int(np.isfinite(group_daily[f"{name} NTL"]).sum()),
        median_SC_pct=np.nanmedian(sc),
        median_floor_pct=np.nanmedian(group_daily[f"{name} floor %"]),
        median_daily_NTL=np.nanmedian(group_daily[f"{name} NTL"]),
    ))
    offset += len(indices)
display(pd.DataFrame(group_summary).round(2))
group_daily.rename_axis("date").to_csv(TABLE_DIR / "dark_bright_daily.csv")


fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    row_heights=[.16, .42, .42], vertical_spacing=.07,
    subplot_titles=["Spatial completeness of fixed pixels",
                    "Bright urban pixels", "Dark pixels"])
group_order = ["Bright urban", "Dark donor", "Dark holdout"]
fig.add_trace(go.Heatmap(
    x=group_daily.index, y=group_order,
    z=np.vstack([group_daily[f"{name} SC"].to_numpy()
                 for name in group_order]),
    colorscale=SC_GREEN_COLORSCALE, zmin=0, zmax=100,
    colorbar=dict(title=dict(text="SC (%)", font=dict(size=16)),
                  thickness=18, tickfont=dict(size=15),
                  x=1.01, y=.92, len=.16),
    hovertemplate="%{y}<br>%{x|%d %b %Y}<br>SC %{z:.1f}%<extra></extra>",
), row=1, col=1)
for name, color in [("Bright urban", "#E95A3D"),
                    ("Dark donor", "#0072B2"),
                    ("Dark holdout", "#56B4E9")]:
    fig.add_trace(go.Scatter(
        x=group_daily.index, y=group_daily[f"{name} NTL"],
        mode="lines+markers", connectgaps=False,
        line=dict(color=color, width=1.1),
        marker=dict(color=color, size=3.5),
        name=name,
        hovertemplate=("%{x|%d %b %Y}<br>NTL %{y:.2f}"
                       "<extra></extra>"),
    ), row=2 if name == "Bright urban" else 3, col=1)
style_figure(fig, "Observed source radiance at fixed pixels", 1550, 900)
fig.update_xaxes(range=[CHARACTERIZATION_START, PRE_EVENT_END],
                 dtick="M3", tickformat="%b %Y", row=3, col=1)
fig.update_yaxes(title_text=None, showticklabels=False, row=1, col=1)
for row in (2, 3):
    fig.update_yaxes(title_text="NTL (nW cm⁻² sr⁻¹)",
                     rangemode="tozero", row=row, col=1)
finish_figure(fig, "04_fixed_pixel_daily_signals")


## 6. Estimate candidate periods from the observed data

Two frequency views are kept distinct. **Lomb–Scargle** uses only dates with observed reliability-qualified radiance, so it does not require filling gaps. The **FFT** uses a regular daily series made by linear interpolation solely for this diagnostic; its peaks can be distorted by missingness. Both inputs have a linear trend removed first. We also examine the SC time series and the binary observation calendar. Peaks shared with SC or the observation window may be sampling artefacts. The strongest *regional observed-only* peak in the 8–60-day band becomes an exploratory daily STL period; the full plot extends to 120 days to expose slower structure.

The ~16-day viewing-geometry and ~29.53-day synodic-month lines are **hypothesis markers**, not imposed model periods. Peak stability is checked separately for 2012 and 2013. If the seasonal signal moves between periods or is concentrated in SC, do not interpret one STL component as a stable physical cycle.

In [ ]:
from scipy.signal import lombscargle, periodogram, detrend
from statsmodels.tsa.seasonal import STL

PERIOD_GRID = np.linspace(8, 120, 1400)
FREQUENCIES = 2 * np.pi / PERIOD_GRID

def observed_spectrum(values, dates_index, periods=PERIOD_GRID):
    series = pd.Series(values, index=dates_index, dtype=float)
    good = series.notna()
    if good.sum() < 40 or series[good].std() < 1e-8:
        return np.full(len(periods), np.nan)
    t = (series.index[good] - series.index[0]).days.to_numpy(dtype=float)
    y = series[good].to_numpy(dtype=float)
    design = np.column_stack([np.ones(len(t)), t])
    residual = y - design @ np.linalg.lstsq(design, y, rcond=None)[0]
    return lombscargle(t, residual, 2*np.pi/periods,
                       precenter=True, normalize=True)

def strongest(periods, power, lo=8, hi=60):
    inside = (periods >= lo) & (periods <= hi) & np.isfinite(power)
    return float(periods[inside][np.argmax(power[inside])]) if inside.any() else np.nan


def harmonic_amplitude(series, period):
    time_days = (series.index - daily_calendar[0]).days.to_numpy(dtype=float)
    phase = 2*np.pi*time_days/period
    design = np.column_stack([
        np.ones(len(time_days)), time_days, np.sin(phase), np.cos(phase)
    ])
    beta = np.linalg.lstsq(design, series.to_numpy(dtype=float), rcond=None)[0]
    return float(np.hypot(beta[2], beta[3]))

regional_log = np.log1p(regional_daily_ntl)
spectra = {
    "Regional RQ daily": observed_spectrum(regional_log, daily_calendar),
    "Spatial completeness": observed_spectrum(regional_daily_sc, daily_calendar),
    "Observation calendar": observed_spectrum(
        regional_daily_ntl.notna().astype(float), daily_calendar),
}
for name in pixel_groups:
    series = np.log1p(group_daily[f"{name} NTL"].reindex(daily_calendar))
    spectra[name] = observed_spectrum(series, daily_calendar)

interpolated_for_fft = regional_log.interpolate(method="time",
    limit_direction="both")
fft_frequency, fft_power = periodogram(
    detrend(interpolated_for_fft.to_numpy()), fs=1.0, window="hann",
    scaling="spectrum")
positive = fft_frequency > 0
fft_periods = 1 / fft_frequency[positive]
fft_spectrum = fft_power[positive]
annual_parts = {}
for year in (2012, 2013):
    part = regional_log.loc[regional_log.index.year == year]
    annual_parts[year] = observed_spectrum(part, part.index)

dominant_period_days = strongest(PERIOD_GRID, spectra["Regional RQ daily"])
STL_PERIOD_DAYS = int(round(dominant_period_days))
peak_rows = [
    dict(signal=name, peak_8_to_60_days=strongest(PERIOD_GRID, power),
         peak_8_to_120_days=strongest(PERIOD_GRID, power, 8, 120))
    for name, power in spectra.items()
]
peak_rows.extend([
    dict(signal="Regional 2012", peak_8_to_60_days=strongest(PERIOD_GRID, annual_parts[2012])),
    dict(signal="Regional 2013", peak_8_to_60_days=strongest(PERIOD_GRID, annual_parts[2013])),
    dict(signal="FFT on interpolated regional",
         peak_8_to_60_days=strongest(fft_periods, fft_spectrum)),
])
peak_table = pd.DataFrame(peak_rows)
display(peak_table.round(1))
peak_table.to_csv(TABLE_DIR / "candidate_periods.csv", index=False)
print(f"Exploratory STL period: {STL_PERIOD_DAYS} daily samples")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    row_heights=[.62, .38], vertical_spacing=.10,
    subplot_titles=["Observed-only radiance spectra",
                    "Sampling and interpolation diagnostics"])
for name, color in [
    ("Regional RQ daily", "#243B5A"), ("Bright urban", "#E95A3D"),
    ("Dark donor", "#0072B2"), ("Dark holdout", "#56B4E9"),
]:
    fig.add_trace(go.Scatter(
        x=PERIOD_GRID, y=spectra[name], mode="lines",
        line=dict(color=color, width=2.3 if name == "Regional RQ daily" else 1.5),
        name=name), row=1, col=1)
for name, color in [
    ("Spatial completeness", "#008A3A"),
    ("Observation calendar", "#9A7A31"),
]:
    fig.add_trace(go.Scatter(
        x=PERIOD_GRID, y=spectra[name], mode="lines",
        line=dict(color=color, width=1.8), name=name), row=2, col=1)
fft_scaled = fft_spectrum / np.nanmax(fft_spectrum)
fig.add_trace(go.Scatter(
    x=fft_periods, y=fft_scaled, mode="lines",
    line=dict(color="#B33D6A", width=1.6, dash="dot"),
    name="FFT · interpolated (scaled)"), row=2, col=1)
for period, label, color in [(16, "16 d · geometry?", "#888888"),
                              (29.53, "29.53 d · lunar?", "#B36C00"),
                              (dominant_period_days, "Detected peak", "#0057FF")]:
    for row in (1, 2):
        fig.add_vline(x=period, row=row, col=1, line_color=color,
                      line_dash="dash", opacity=.7)
style_figure(fig, "Pre-Haiyan candidate periods", 1550, 820)
fig.update_xaxes(range=[8, 120], title_text="Period (days)", row=2, col=1)
fig.update_yaxes(title_text="Normalized power", row=1, col=1)
fig.update_yaxes(title_text="Relative diagnostic power", row=2, col=1)
finish_figure(fig, "05_observed_and_fft_periods")

### 6a. Could the observation calendar manufacture the detected peak?

The regional peak above was chosen from observed-only radiance. That still does not make it physical: irregular sampling can put side-lobes around a real frequency. Here two *known* synthetic waves (16 and 29.53 days) are sampled on exactly the same days as qualified regional NTL. Their spectra show where this observation window would move or replicate power. Compare those patterns with the actual NTL, SC and observed-day spectra before naming an STL period. The FFT is a cross-check only because it uses interpolated data.

In [ ]:
observed_mask = regional_daily_ntl.notna().to_numpy()
elapsed_days = (daily_calendar - daily_calendar[0]).days.to_numpy(dtype=float)
synthetic_spectra = {}
for true_period in (16.0, 29.53):
    signal = np.sin(2*np.pi*elapsed_days/true_period)
    signal[~observed_mask] = np.nan
    synthetic_spectra[true_period] = observed_spectrum(
        signal, daily_calendar
    )

period_checks = []
for period in [16.0, 28.0, 29.53, dominant_period_days]:
    nearest = int(abs(PERIOD_GRID - period).argmin())
    period_checks.append({
        "period_days": round(period, 2),
        "RQ_power": spectra["Regional RQ daily"][nearest],
        "SC_power": spectra["Spatial completeness"][nearest],
        "observation_calendar_power": spectra["Observation calendar"][nearest],
        "16_day_wave_after_sampling": synthetic_spectra[16.0][nearest],
        "29_53_day_wave_after_sampling": synthetic_spectra[29.53][nearest],
    })
period_checks = pd.DataFrame(period_checks)
display(period_checks.round(3))
period_checks.to_csv(TABLE_DIR / "periods_vs_observation_window.csv", index=False)

year_peaks = peak_table.set_index("signal")["peak_8_to_60_days"]
year_gap = abs(year_peaks["Regional 2012"] - year_peaks["Regional 2013"])
print(f"2012 versus 2013 regional peak separation: {year_gap:.1f} days.")
print("A large separation, or a peak shared mainly with SC/the sampling window, "
      "weakens a stable seasonal interpretation.")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    row_heights=[.56, .44], vertical_spacing=.10,
    subplot_titles=["Observed regional NTL and observation window",
                    "Known cycles passed through the same observation calendar"])
for name, color in [
    ("Regional RQ daily", "#243B5A"),
    ("Spatial completeness", "#008A3A"),
    ("Observation calendar", "#9A7A31"),
]:
    fig.add_trace(go.Scatter(
        x=PERIOD_GRID, y=spectra[name], mode="lines",
        line=dict(color=color, width=2.2 if name == "Regional RQ daily" else 1.5),
        name=name,
    ), row=1, col=1)
for true_period, color in [(16.0, "#0072B2"), (29.53, "#CC79A7")]:
    fig.add_trace(go.Scatter(
        x=PERIOD_GRID, y=synthetic_spectra[true_period], mode="lines",
        line=dict(color=color, width=2),
        name=f"{true_period:g}-day input · observed dates",
    ), row=2, col=1)
for test_period in (16.0, 29.53, dominant_period_days):
    for row in (1, 2):
        fig.add_vline(
            x=test_period, row=row, col=1, line_color=PLOT_GRID_COLOR,
            line_dash="dash", line_width=1,
        )
style_figure(fig, "Sampling-window alias check", 1550, 800)
fig.update_xaxes(range=[8, 70], title_text="Period (days)", row=2, col=1)
fig.update_yaxes(title_text="Normalized power", row=1, col=1)
fig.update_yaxes(title_text="Normalized power", row=2, col=1)
finish_figure(fig, "05a_sampling_window_alias")

### 6b. Is the candidate frequency coherent across individual pixels?

A regional median can oscillate because the contributing pixels change. Fit the detected-period sine and cosine separately to each *observed pixel* (with a linear trend), then examine amplitude and phase. Coherent phase among bright pixels, replicated in both years and not mirrored by the observation calendar, supports a common radiance component. Coherence among floor-bound dark pixels is less informative. This diagnostic does not assume that all pixels have the same absolute amplitude. Phase wraps at ±180°; use the circular concentration value in the table rather than reading a visual split at that boundary as disagreement.

In [ ]:
def fit_pixel_cycle(values, index, period):
    good = np.isfinite(values)
    if good.sum() < 40 or np.nanstd(values) < 1e-8:
        return None
    time = (index[good] - daily_calendar[0]).days.to_numpy(dtype=float)
    y = values[good].astype(float)
    angle = 2*np.pi*time/period
    base = np.column_stack([np.ones(len(time)), time])
    full = np.column_stack([base, np.sin(angle), np.cos(angle)])
    beta = np.linalg.lstsq(full, y, rcond=None)[0]
    base_beta = np.linalg.lstsq(base, y, rcond=None)[0]
    full_error = np.sum((y - full @ beta)**2)
    base_error = np.sum((y - base @ base_beta)**2)
    return dict(
        observed_days=int(good.sum()),
        amplitude=float(np.hypot(beta[2], beta[3])),
        phase_rad=float(np.arctan2(beta[3], beta[2])),
        added_fit_fraction=float(
            max(0, 1 - full_error/base_error) if base_error > 0 else np.nan
        ),
        floor_share_pct=float(
            100*np.isclose(y, .01, atol=.001).mean()
        ),
    )

pixel_cycle_rows = []
for column, pixel in enumerate(selected_pixels):
    for period_label, use_dates in [
        ("All", np.ones(len(selected_dates), dtype=bool)),
        ("2012", selected_dates.year == 2012),
        ("2013", selected_dates.year == 2013),
    ]:
        result = fit_pixel_cycle(
            pixel_values[use_dates, column],
            selected_dates[use_dates], dominant_period_days,
        )
        if result is not None:
            pixel_cycle_rows.append(dict(
                pixel_index=int(pixel), group=group_id[column],
                period_label=period_label, NTL0=float(ntl0[pixel]),
                **result,
            ))
pixel_cycle = pd.DataFrame(pixel_cycle_rows)
pixel_cycle.to_csv(TABLE_DIR / "individual_pixel_cycle.csv", index=False)

def phase_concentration(phase_values):
    return float(abs(np.mean(np.exp(1j*np.asarray(phase_values)))))

coherence = pixel_cycle.groupby(["group", "period_label"]).agg(
    pixels=("pixel_index", "nunique"),
    median_amplitude=("amplitude", "median"),
    median_added_fit=("added_fit_fraction", "median"),
    median_observed_days=("observed_days", "median"),
    median_floor_pct=("floor_share_pct", "median"),
    phase_concentration=("phase_rad", phase_concentration),
).reset_index()
display(coherence.round(3))
coherence.to_csv(TABLE_DIR / "pixel_cycle_coherence.csv", index=False)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=.14,
    subplot_titles=["Amplitude versus reference radiance",
                    "Phase of the detected cycle"])
colors = {
    "Bright urban": "#E95A3D",
    "Dark donor": "#0072B2",
    "Dark holdout": "#56B4E9",
}
all_period = pixel_cycle.loc[pixel_cycle.period_label.eq("All")]
for name, color in colors.items():
    part = all_period.loc[all_period.group.eq(name)]
    common_hover = np.column_stack([
        part.observed_days, part.floor_share_pct,
        part.added_fit_fraction,
    ])
    fig.add_trace(go.Scatter(
        x=part.NTL0, y=part.amplitude,
        mode="markers", marker=dict(color=color, size=8, opacity=.75),
        name=name, customdata=common_hover,
        hovertemplate=("%{x:.2f} NTL₀<br>Amplitude %{y:.3f}"
                       "<br>Observed days %{customdata[0]:.0f}"
                       "<br>Floor %{customdata[1]:.1f}%"
                       "<br>Added fit %{customdata[2]:.2f}<extra></extra>"),
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=part.NTL0, y=np.rad2deg(part.phase_rad),
        mode="markers", marker=dict(color=color, size=8, opacity=.75),
        name=name, showlegend=False, customdata=common_hover,
        hovertemplate=("%{x:.2f} NTL₀<br>Phase %{y:.1f}°"
                       "<br>Observed days %{customdata[0]:.0f}"
                       "<br>Floor %{customdata[1]:.1f}%<extra></extra>"),
    ), row=1, col=2)
style_figure(fig, "Individual-pixel cycle support", 1500, 610)
fig.update_xaxes(title_text="60-day reference NTL₀ (nW cm⁻² sr⁻¹)",
                 row=1, col=1)
fig.update_xaxes(title_text="60-day reference NTL₀ (nW cm⁻² sr⁻¹)",
                 row=1, col=2)
fig.update_yaxes(title_text="Harmonic amplitude (nW cm⁻² sr⁻¹)",
                 row=1, col=1)
fig.update_yaxes(title_text="Phase (degrees)", range=[-180, 180],
                 row=1, col=2)
finish_figure(fig, "05b_pixel_cycle_coherence")

## 7. Decompose the daily series using the detected period

This STL is a **description** of the interpolated daily log1p regional NTL, not a causal model or a post-Haiyan forecast. Interpolation supplies the regular grid STL requires; observed values and interpolated-only input are therefore distinguished explicitly. STL estimates a smooth trend, a repeating component at the detected period, and a residual. That decomposition cannot tell whether the repeating component comes from electrification, atmospheric/viewing effects, or changing visible-pixel composition. Compare its peaks and troughs with the SC strip and the fixed dark/bright groups above.

In [ ]:
stl_input = regional_log.interpolate(method="time", limit_direction="both")
stl_fit = STL(stl_input, period=STL_PERIOD_DAYS,
              robust=True).fit()
stl_fitted_log = stl_fit.trend + stl_fit.seasonal
fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
    row_heights=[.10, .40, .25, .25], vertical_spacing=.055,
    subplot_titles=["Regional SC", "Observed RQ NTL and STL fit",
                    "STL trend · log1p NTL",
                    f"STL {STL_PERIOD_DAYS}-day component and residual · log1p NTL"])
fig.add_trace(go.Heatmap(
    x=daily_calendar, y=["SC"], z=[regional_daily_sc.to_numpy()],
    colorscale=SC_GREEN_COLORSCALE, zmin=0, zmax=100,
    colorbar=dict(title="SC (%)", thickness=16, len=.15, y=.92),
    showscale=True), row=1, col=1)
fig.add_trace(go.Scatter(
    x=daily_calendar, y=regional_daily_ntl, mode="markers",
    marker=dict(color=RQ_COLOR, size=4), name="Observed RQ NTL (daily)"),
    row=2, col=1)
fig.add_trace(go.Scatter(
    x=daily_calendar, y=np.expm1(stl_fitted_log), mode="lines",
    line=dict(color=BASELINE_COLOR, width=1.8), name="STL fit"),
    row=2, col=1)
missing_input = ~regional_daily_ntl.notna()
fig.add_trace(go.Scatter(
    x=daily_calendar[missing_input], y=np.expm1(stl_input[missing_input]),
    mode="markers", marker=dict(color="#B9C0CA", size=4, symbol="circle-open"),
    name="Interpolated STL input"), row=2, col=1)
fig.add_trace(go.Scatter(
    x=daily_calendar, y=stl_fit.trend, mode="lines",
    line=dict(color="#243B5A", width=2.4), name="Trend"),
    row=3, col=1)
for values, color, name, mode in [
    (stl_fit.seasonal, "#009E73", f"{STL_PERIOD_DAYS}-day component", "lines"),
    (stl_fit.resid, "#CC79A7", "Residual", "markers"),
]:
    fig.add_trace(go.Scatter(
        x=daily_calendar, y=values, mode=mode,
        line=dict(color=color, width=1.3),
        marker=dict(color=color, size=3, opacity=.55),
        name=name), row=4, col=1)
fig.add_hline(y=0, line_color=BASELINE_COLOR, line_width=1,
              row=4, col=1)
style_figure(fig, "Observed-data-led STL diagnostic", 1550, 1020)
fig.update_yaxes(title_text="NTL", row=2, col=1)
fig.update_yaxes(title_text="log1p NTL", row=3, col=1)
fig.update_yaxes(title_text="log1p NTL", row=4, col=1)
fig.update_xaxes(dtick="M3", tickformat="%b %Y", row=4, col=1)
finish_figure(fig, "06_detected_period_stl")

## 8. Test whether viewing geometry explains the regional cycle

VNP46A1 supplies `Sensor_Zenith` and, when present, `Sensor_Azimuth`; VNP46A2 supplies the corrected DNB-BRDF radiance and MQF. Residual angular effects and changing spatial support have been reported even in corrected nighttime-lights series ([NASA VNP46A1](https://ladsweb.modaps.eosdis.nasa.gov/missions-and-measurements/products/VNP46A1); [angular-effect study](https://www.sciencedirect.com/science/article/pii/S0034425724000889)). The question here is whether they explain **this regional G3 baseline**, not whether a correction can make a line smoother.

First, the calendar-phase plots are proxies only. Next, the notebook pairs measured A1 angles with MQF=0 A2 radiance at the **same date and pixel**. The investigation expands from all fixed regional G3 pixels to GHSL classes 22, 23 and 30, then tests a 2012-trained angle response on 2013. Bright and dark selected pixels follow as independent controls. Paired spatial completeness and the match against the original RQ regional series stay visible throughout. No 50% VZA gate is introduced.

The two A1 stores are opened once in the next cell. The selected-pixel CSV is an output, never an input, so an older May-2013 file cannot truncate the 2012 plots.


In [ ]:
# Phase is a calendar proxy only. It is never labeled as observed VZA.
phase_frame = pd.DataFrame(index=daily_calendar)
phase_frame["year"] = daily_calendar.year
phase_frame["phase"] = ((daily_calendar - daily_calendar[0]).days % 16)
phase_frame["regional_NTL"] = regional_daily_ntl
phase_frame["regional_SC"] = regional_daily_sc
phase_frame["bright_NTL"] = group_daily["Bright urban NTL"].reindex(daily_calendar)
phase_frame["dark_NTL"] = group_daily["Dark donor NTL"].reindex(daily_calendar)
regional_uncapped = baseline_daily.set_index("date").raw_uncapped.reindex(daily_calendar)
phase_frame["low_qualified"] = (
    regional_uncapped.le(LOW_NTL) & regional_daily_ntl.notna()
).astype(int)
phase_frame["qualified"] = regional_daily_ntl.notna().astype(int)

phase_year = phase_frame.groupby(["year", "phase"]).agg(
    regional_NTL=("regional_NTL", "median"),
    bright_NTL=("bright_NTL", "median"),
    dark_NTL=("dark_NTL", "median"),
    median_SC=("regional_SC", "median"),
    qualified_days=("qualified", "sum"),
    low_days=("low_qualified", "sum"),
)
phase_year["near_zero_rate_pct"] = (
    100*phase_year.low_days/phase_year.qualified_days.replace(0, np.nan)
)
for name in ("regional_NTL", "bright_NTL", "dark_NTL"):
    phase_year[name + "_anomaly"] = (
        phase_year[name] -
        phase_year.groupby(level="year")[name].transform("median")
    )
display(phase_year.round(2))
phase_year.to_csv(TABLE_DIR / "sixteen_day_phase_by_year.csv")

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    row_heights=[.16, .48, .36], vertical_spacing=.08,
    subplot_titles=["Regional SC by calendar phase",
                    "Regional NTL anomaly · independent years",
                    "Near-zero share among qualified days"])
years = [2012, 2013]
sc_grid = np.vstack([
    phase_year.loc[year].reindex(range(16)).median_SC.to_numpy()
    for year in years
])
fig.add_trace(go.Heatmap(
    x=list(range(16)), y=[str(year) for year in years], z=sc_grid,
    colorscale=SC_GREEN_COLORSCALE, zmin=0, zmax=100,
    colorbar=dict(title=dict(text="SC (%)", font=dict(size=16)),
                  thickness=18, tickfont=dict(size=15),
                  x=1.01, y=.92, len=.16),
    hovertemplate="%{y} · phase %{x}<br>SC %{z:.1f}%<extra></extra>",
), row=1, col=1)
for year, color in [(2012, "#243B5A"), (2013, "#0072B2")]:
    part = phase_year.loc[year].reindex(range(16))
    fig.add_trace(go.Scatter(
        x=part.index, y=part.regional_NTL_anomaly,
        mode="lines+markers", line=dict(color=color, width=2),
        marker=dict(size=5), name=str(year) + " RQ NTL",
    ), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=part.index, y=part.near_zero_rate_pct,
        mode="lines+markers", line=dict(color=color, width=1.8,
                                       dash="dash" if year == 2013 else "solid"),
        marker=dict(size=5), name=str(year) + " low-day share",
        customdata=np.column_stack([part.low_days, part.qualified_days]),
        hovertemplate=("Phase %{x}<br>Near-zero %{y:.1f}%"
                       "<br>%{customdata[0]:.0f} / %{customdata[1]:.0f}"
                       " qualified days<extra></extra>"),
    ), row=3, col=1)
style_figure(fig, "Sixteen-day calendar-phase screen", 1450, 830)
fig.update_xaxes(tickmode="array", tickvals=list(range(16)),
                 title_text="Day within the 16-day calendar cycle",
                 row=3, col=1)
fig.update_yaxes(showticklabels=False, row=1, col=1)
fig.update_yaxes(title_text="NTL anomaly (nW cm⁻² sr⁻¹)", row=2, col=1)
fig.update_yaxes(title_text="Near-zero (%)", rangemode="tozero",
                 row=3, col=1)
finish_figure(fig, "08a_sixteen_day_regional_phase")

fig = make_subplots(rows=1, cols=2, horizontal_spacing=.12,
    subplot_titles=["Bright urban pixels", "Dark donor pixels"])
for col, name, color in [
    (1, "bright_NTL_anomaly", "#E95A3D"),
    (2, "dark_NTL_anomaly", "#0072B2"),
]:
    for year, dash in [(2012, "solid"), (2013, "dash")]:
        part = phase_year.loc[year].reindex(range(16))
        fig.add_trace(go.Scatter(
            x=part.index, y=part[name], mode="lines+markers",
            line=dict(color=color, width=2, dash=dash),
            marker=dict(size=5), name=f"{year} · {name.split('_')[0]}",
            showlegend=(col == 1),
        ), row=1, col=col)
    fig.update_xaxes(tickmode="array", tickvals=list(range(0, 16, 2)),
                     title_text="Calendar phase (days)", row=1, col=col)
    fig.update_yaxes(title_text="NTL anomaly (nW cm⁻² sr⁻¹)",
                     row=1, col=col)
style_figure(fig, "Fixed-pixel 16-day phase · year-to-year check", 1450, 560)
finish_figure(fig, "08b_sixteen_day_fixed_pixels")

In [ ]:
# Measured VZA: assemble the full pre-Haiyan A1 record once.
# Never prefer an older selected-pixel CSV over these stores.
a1_early = xr.open_zarr(
    PROCESSED_DIR / "baseline/Haiyan_VNP46A1_baseline.zarr",
    chunks={}, consolidated=False, mask_and_scale=False,
)
a1_later = xr.open_zarr(
    PROCESSED_DIR / "Haiyan_VNP46A1.zarr",
    chunks={}, consolidated=False, mask_and_scale=False,
)
for axis in ("x", "y"):
    np.testing.assert_allclose(
        a1_early[axis].values, a1_later[axis].values,
        rtol=0, atol=1e-6,
    )
    np.testing.assert_allclose(
        a1_later[axis].values, a2[axis].values,
        rtol=0, atol=1e-6,
    )
a1_early = a1_early.assign_coords(x=a1_later.x, y=a1_later.y)
if "spatial_ref" in a1_later:
    a1_early["spatial_ref"] = a1_later["spatial_ref"]

early_dates = pd.DatetimeIndex(a1_early.date.values)
later_dates = pd.DatetimeIndex(a1_later.date.values)
early_selected = selected_dates.intersection(early_dates).difference(later_dates)
later_selected = selected_dates.intersection(later_dates)
a1_parts = []
if len(early_selected):
    a1_parts.append(a1_early.sel(date=early_selected))
if len(later_selected):
    a1_parts.append(a1_later.sel(date=later_selected))
if not a1_parts:
    raise ValueError("Neither A1 store overlaps the selected baseline dates.")
a1 = xr.concat(
    a1_parts, dim="date", data_vars="minimal", coords="minimal",
    compat="equals", join="exact",
).sortby("date")

a1_dates = pd.DatetimeIndex(a1.date.values)
missing_a1 = selected_dates.difference(a1_dates)
if len(missing_a1):
    raise ValueError(
        f"A1 misses {len(missing_a1)} selected dates; "
        f"first missing: {missing_a1[0]:%Y-%m-%d}"
    )
if "Sensor_Zenith" not in a1:
    raise KeyError("The combined A1 record has no Sensor_Zenith band.")
print(
    f"Combined A1: {a1_dates.min():%Y-%m-%d} to "
    f"{a1_dates.max():%Y-%m-%d} "
    f"({len(a1_dates)} of {len(selected_dates)} selected dates)"
)

def extract_selected_a1_angle(
    band_name, valid_min, valid_max, sample_y=None, sample_x=None
):
    sample_y = selected_y if sample_y is None else sample_y
    sample_x = selected_x if sample_x is None else sample_x
    band = a1[band_name]
    result = np.full(
        (len(selected_dates), len(sample_y)), np.nan, dtype=float
    )
    for start in tqdm(
        range(0, len(selected_dates), 32),
        desc=f"Select {band_name} on fixed pixels",
    ):
        sl = slice(start, min(start + 32, len(selected_dates)))
        sampled = band.sel(date=selected_dates[sl]).isel(
            y=xr.DataArray(sample_y, dims="sample_pixel"),
            x=xr.DataArray(sample_x, dims="sample_pixel"),
        )
        extra = {
            dim: 0 for dim in sampled.dims
            if dim not in ("date", "sample_pixel")
        }
        if any(sampled.sizes[dim] != 1 for dim in extra):
            raise ValueError(f"Unexpected non-singleton {band_name} dimension")
        if extra:
            sampled = sampled.isel(extra, drop=True)
        result[sl] = (
            sampled.transpose("date", "sample_pixel")
            .compute().values.astype(float)
        )

    fill_values = list(KNOWN_FILL_VALUES)
    for key in ("_FillValue", "missing_value"):
        value = band.attrs.get(key)
        if value is not None:
            fill_values.extend(np.atleast_1d(value).tolist())
    for value in fill_values:
        try:
            result[result == float(value)] = np.nan
        except (TypeError, ValueError):
            pass

    scale = float(band.attrs.get("scale_factor", 1.0))
    offset = float(band.attrs.get("add_offset", 0.0))
    result = result * scale + offset
    finite = np.abs(result[np.isfinite(result)])
    if finite.size and np.percentile(finite, 95) > valid_max:
        result *= .01  # A1 angles stored as centidegrees without metadata.
    result[(result < valid_min) | (result > valid_max)] = np.nan
    return result

zenith_values = extract_selected_a1_angle("Sensor_Zenith", 0, 90)
angle = pd.DataFrame({
    "date": np.repeat(selected_dates.to_numpy(), len(selected_pixels)),
    "pixel_index": np.tile(selected_pixels, len(selected_dates)),
    "sensor_zenith_deg": zenith_values.ravel(),
})
if "Sensor_Azimuth" in a1:
    azimuth_values = extract_selected_a1_angle(
        "Sensor_Azimuth", -360, 360
    )
    angle["sensor_azimuth_deg"] = azimuth_values.ravel()

pixel_long = pd.DataFrame({
    "date": np.repeat(selected_dates.to_numpy(), len(selected_pixels)),
    "pixel_index": np.tile(selected_pixels, len(selected_dates)),
    "ntl": pixel_values.ravel(),
    "group": np.tile(group_id, len(selected_dates)),
}).dropna(subset=["ntl"])
paired_angle = pixel_long.merge(
    angle, on=["date", "pixel_index"], how="inner",
    validate="one_to_one",
).dropna(subset=["sensor_zenith_deg"]).copy()
paired_angle["year"] = paired_angle.date.dt.year
paired_angle["month"] = paired_angle.date.dt.to_period("M")
paired_angle["ntl_within_pixel"] = (
    paired_angle.ntl
    - paired_angle.groupby("pixel_index").ntl.transform("median")
)

support_by_year = paired_angle.groupby(["year", "group"]).agg(
    pairs=("ntl", "size"),
    pixels=("pixel_index", "nunique"),
    dates=("date", "nunique"),
).reset_index()
display(support_by_year)
if not paired_angle.year.eq(2012).any():
    raise ValueError(
        "A1 dates include 2012 but no same-pixel 2012 VZA/MQF=0 "
        "NTL pairs survived. Check angle scale and fill values."
    )

# The CSV is an output of this complete extract, never an input to it.
vza_path = NB8_DIR / "inputs/sensor_zenith_selected_pixels.csv"
vza_path.parent.mkdir(parents=True, exist_ok=True)
angle.to_csv(vza_path, index=False)


### 8a. Is the regional dip present on the same pixel-days as high VZA?

Use **every fixed Regional G3 pixel**, not just selected controls. Match A1 angle to finite MQF=0 A2 radiance, then show the original RQ regional median, the median on VZA-matched pixels, median VZA and paired SC over the complete baseline at both daily and four-day resolution. The uncapped MQF=0 median isolates the effect of A1 matching; the RQ line additionally reflects the existing P95/SC route. A difference between these lines cannot be credited to viewing angle.


In [ ]:
regional_pixels = fixed["Regional G3"]
regional_flat = land_pixels[regional_pixels]
regional_y, regional_x = np.unravel_index(regional_flat, land.shape)
regional_classes = ghsl_vector[regional_pixels].astype(int)
regional_vza = extract_selected_a1_angle(
    "Sensor_Zenith", 0, 90,
    sample_y=regional_y, sample_x=regional_x,
).astype("float32")

regional_source_ntl = np.full(regional_vza.shape, np.nan, dtype="float32")
for start in tqdm(
    range(0, len(selected_dates), 32),
    desc="Match regional G3 source pixels",
):
    sl = slice(start, min(start + 32, len(selected_dates)))
    batch = a2[[DNB_BAND, MQF_BAND]].sel(
        date=selected_dates[sl]
    ).isel(
        y=xr.DataArray(regional_y, dims="sample_pixel"),
        x=xr.DataArray(regional_x, dims="sample_pixel"),
    ).compute()
    values = batch[DNB_BAND].transpose(
        "date", "sample_pixel"
    ).values.astype("float32")
    quality = batch[MQF_BAND].transpose(
        "date", "sample_pixel"
    ).values
    valid = (
        (quality == 0) & np.isfinite(values) & (values >= 0)
        & ~np.isin(values, KNOWN_FILL_VALUES)
    )
    regional_source_ntl[sl] = np.where(valid, values, np.nan)

regional_pair = (
    np.isfinite(regional_source_ntl) & np.isfinite(regional_vza)
)
regional_matched_ntl = np.where(
    regional_pair, regional_source_ntl, np.nan
)
regional_matched_vza = np.where(
    regional_pair, regional_vza, np.nan
)
regional_pair_count = regional_pair.sum(axis=1)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    regional_vza_q25 = np.nanpercentile(
        regional_matched_vza, 25, axis=1
    )
    regional_vza_q75 = np.nanpercentile(
        regional_matched_vza, 75, axis=1
    )

regional_angle_daily = pd.DataFrame({
    "date": selected_dates,
    "RQ_regional_NTL": regional_daily_ntl.reindex(
        selected_dates
    ).to_numpy(),
    "raw_uncapped_NTL": baseline_daily.set_index("date").raw_uncapped.reindex(
        selected_dates
    ).to_numpy(),
    "matched_NTL": median(regional_matched_ntl, axis=1),
    "median_VZA": median(regional_matched_vza, axis=1),
    "VZA_q25": regional_vza_q25,
    "VZA_q75": regional_vza_q75,
    "paired_pixels": regional_pair_count,
    "paired_SC_pct": 100 * regional_pair_count / len(regional_pixels),
})
regional_angle_daily["year"] = regional_angle_daily.date.dt.year
regional_angle_daily["RQ_comparable"] = (
    regional_angle_daily.RQ_regional_NTL.notna()
    & regional_angle_daily.paired_SC_pct.ge(SPATIAL_COMPLETENESS_PCT)
)
regional_angle_daily["matched_minus_RQ"] = (
    regional_angle_daily.matched_NTL
    - regional_angle_daily.RQ_regional_NTL
)
regional_angle_daily["matched_minus_raw"] = (
    regional_angle_daily.matched_NTL
    - regional_angle_daily.raw_uncapped_NTL
)
regional_angle_daily.to_csv(
    TABLE_DIR / "regional_g3_daily_vza_matched.csv", index=False
)

# Match the notebook's four-day pixel-then-spatial aggregation order.
regional_block_id = np.asarray(
    (selected_dates - EVENT_DATE).days // AGGREGATION_DAYS
)
regional_4d_rows = []
for block in baseline_4d.itertuples():
    block_id = int((block.date - EVENT_DATE).days // AGGREGATION_DAYS)
    day_rows = np.flatnonzero(regional_block_id == block_id)
    if not len(day_rows):
        continue
    pixel_ntl_4d = median(regional_matched_ntl[day_rows], axis=0)
    pixel_vza_4d = median(regional_matched_vza[day_rows], axis=0)
    paired_pixel_4d = (
        np.isfinite(pixel_ntl_4d) & np.isfinite(pixel_vza_4d)
    )
    regional_4d_rows.append(dict(
        date=block.date,
        RQ_NTL_4D=block.raw_median_ntl,
        raw_uncapped_NTL_4D=block.raw_uncapped,
        matched_NTL_4D=median(pixel_ntl_4d[paired_pixel_4d]),
        median_VZA_4D=median(pixel_vza_4d[paired_pixel_4d]),
        paired_pixels_4D=int(paired_pixel_4d.sum()),
        paired_SC_4D_pct=(
            100 * paired_pixel_4d.mean()
        ),
    ))
regional_angle_4d = pd.DataFrame(regional_4d_rows)
regional_angle_4d["RQ_comparable"] = (
    regional_angle_4d.RQ_NTL_4D.notna()
    & regional_angle_4d.paired_SC_4D_pct.ge(SPATIAL_COMPLETENESS_PCT)
)
regional_angle_4d.to_csv(
    TABLE_DIR / "regional_g3_four_day_vza_matched.csv", index=False
)
regional_support = regional_angle_daily.groupby("year").agg(
    calendar_days=("date", "size"),
    paired_days=("matched_NTL", "count"),
    RQ_comparable_days=("RQ_comparable", "sum"),
    median_paired_pixels=("paired_pixels", "median"),
    median_paired_SC_pct=("paired_SC_pct", "median"),
    median_abs_match_minus_raw=(
        "matched_minus_raw", lambda x: x.abs().median()
    ),
    median_abs_match_minus_RQ=(
        "matched_minus_RQ", lambda x: x.abs().median()
    ),
)
display(regional_support.round(3))

regional_plot = regional_angle_daily.set_index("date").reindex(
    daily_calendar
)
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    row_heights=[.13, .50, .37], vertical_spacing=.06,
    subplot_titles=[
        "Paired regional G3 spatial completeness",
        "Original RQ NTL versus same-pixel VZA-matched NTL",
        "Measured regional viewing angle",
    ],
)
fig.add_trace(go.Heatmap(
    x=daily_calendar, y=["G3 paired SC"],
    z=[regional_plot.paired_SC_pct.to_numpy()],
    colorscale=SC_GREEN_COLORSCALE, zmin=0, zmax=100,
    colorbar=dict(
        title=dict(text="SC (%)", font=dict(size=16)),
        tickfont=dict(size=15), thickness=18,
        x=1.01, y=.93, len=.14,
    ),
    hovertemplate=(
        "%{x|%d %b %Y}<br>Paired SC %{z:.1f}%"
        "<extra></extra>"
    ),
), row=1, col=1)
for column, name, color in [
    ("raw_uncapped_NTL", "Regional MQF=0, uncapped", "#9AA3AD"),
    ("RQ_regional_NTL", "Regional RQ NTL", RQ_COLOR),
    ("matched_NTL", "VZA-matched NTL", "#243B5A"),
]:
    fig.add_trace(go.Scatter(
        x=daily_calendar, y=regional_plot[column],
        mode="lines+markers", connectgaps=False,
        line=dict(color=color, width=1.25),
        marker=dict(color=color, size=3.5),
        name=name,
        customdata=regional_plot[["paired_pixels", "paired_SC_pct"]],
        hovertemplate=(
            "%{x|%d %b %Y}<br>NTL %{y:.3f}"
            "<br>Paired pixels %{customdata[0]:.0f}"
            "<br>Paired SC %{customdata[1]:.1f}%"
            "<extra></extra>"
        ),
    ), row=2, col=1)
low_days = regional_plot.matched_NTL.le(LOW_NTL)
fig.add_trace(go.Scatter(
    x=daily_calendar[low_days],
    y=regional_plot.loc[low_days, "matched_NTL"],
    mode="markers",
    marker=dict(
        color="#D55E00", size=8, symbol="circle-open",
        line=dict(width=1.6),
    ),
    name=f"Matched NTL ≤ {LOW_NTL:g}",
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=daily_calendar, y=regional_plot.VZA_q25,
    mode="lines", line=dict(color="rgba(0,114,178,0)", width=0),
    showlegend=False, hoverinfo="skip",
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=daily_calendar, y=regional_plot.VZA_q75,
    mode="lines", fill="tonexty",
    fillcolor="rgba(0,114,178,.14)",
    line=dict(color="rgba(0,114,178,0)", width=0),
    name="VZA interquartile range",
    hoverinfo="skip",
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=daily_calendar, y=regional_plot.median_VZA,
    mode="lines+markers", connectgaps=False,
    line=dict(color="#0072B2", width=1.6),
    marker=dict(color="#0072B2", size=3.5),
    name="Median VZA",
), row=3, col=1)
style_figure(fig, "Regional G3 · matched NTL and measured VZA", 1550, 950)
fig.update_yaxes(showticklabels=False, row=1, col=1)
fig.update_yaxes(title_text="NTL (nW cm⁻² sr⁻¹)", row=2, col=1)
fig.update_yaxes(title_text="Sensor zenith (degrees)", row=3, col=1)
fig.update_xaxes(
    range=[CHARACTERIZATION_START, PRE_EVENT_END],
    dtick="M3", tickformat="%b %Y", row=3, col=1,
)
finish_figure(fig, "08c_regional_g3_vza_timeline")

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    row_heights=[.14, .52, .34], vertical_spacing=.07,
    subplot_titles=[
        "Four-day paired SC", "Four-day RQ versus VZA-matched NTL",
        "Median VZA of the same four-day pixel composites",
    ],
)
fig.add_trace(go.Heatmap(
    x=regional_angle_4d.date, y=["4D paired SC"],
    z=[regional_angle_4d.paired_SC_4D_pct.to_numpy()],
    colorscale=SC_GREEN_COLORSCALE, zmin=0, zmax=100,
    colorbar=dict(
        title=dict(text="SC (%)", font=dict(size=16)),
        tickfont=dict(size=15), thickness=18,
        x=1.01, y=.93, len=.15,
    ),
), row=1, col=1)
for column, label, color in [
    ("raw_uncapped_NTL_4D", "MQF=0 uncapped (4D)", "#9AA3AD"),
    ("RQ_NTL_4D", "Regional RQ NTL (4D)", RQ_FOUR_DAY_COLOR),
    ("matched_NTL_4D", "VZA-matched NTL (4D)", "#243B5A"),
]:
    fig.add_trace(go.Scatter(
        x=regional_angle_4d.date, y=regional_angle_4d[column],
        mode="lines+markers", connectgaps=False,
        line=dict(color=color, width=1.8, shape="hv"),
        marker=dict(size=3.5), name=label,
    ), row=2, col=1)
fig.add_trace(go.Scatter(
    x=regional_angle_4d.date,
    y=regional_angle_4d.median_VZA_4D,
    mode="lines+markers", connectgaps=False,
    line=dict(color="#0072B2", width=1.7),
    marker=dict(size=3.5), name="Median VZA (4D)",
), row=3, col=1)
style_figure(fig, "Regional G3 · four-day VZA match", 1550, 810)
fig.update_yaxes(showticklabels=False, row=1, col=1)
fig.update_yaxes(title_text="NTL (nW cm⁻² sr⁻¹)", row=2, col=1)
fig.update_yaxes(title_text="Sensor zenith (degrees)", row=3, col=1)
fig.update_xaxes(
    range=[CHARACTERIZATION_START, PRE_EVENT_END],
    dtick="M3", tickformat="%b %Y", row=3, col=1,
)
finish_figure(fig, "08c_regional_g3_four_day_vza")


### 8b. Does regional VZA share the dip frequency?

For inference, use days meeting the established 10% RQ SC threshold in both the original and VZA-matched series; the timeline above still displays all matched days. Compare observed-only spectra of regional matched NTL, median VZA and paired SC. Then inspect NTL versus VZA by year, with SC and paired-pixel count in hover. A common peak or scatter relationship can motivate the pixel-level test, but it does **not** establish causation: viewing angle, cloud selection and contributing-pixel composition can recur together.


In [ ]:
regional_spectra = {}
regional_spectrum_days = {}
for name, series in [
    ("Matched G3 NTL", np.log1p(regional_plot.matched_NTL.where(
        regional_plot.RQ_comparable.fillna(False)
    ))),
    ("Median VZA", regional_plot.median_VZA.where(
        regional_plot.RQ_comparable.fillna(False)
    )),
    ("Paired SC", regional_plot.paired_SC_pct),
]:
    regional_spectra[name] = observed_spectrum(
        series, daily_calendar
    )
    regional_spectrum_days[name] = int(series.notna().sum())

regional_period_rows = []
for name, power in regional_spectra.items():
    regional_period_rows.append(dict(
        signal=name,
        peak_8_to_60_days=strongest(PERIOD_GRID, power),
        power_at_16_days=power[int(abs(PERIOD_GRID - 16).argmin())],
        observed_days=regional_spectrum_days[name],
    ))
regional_periods = pd.DataFrame(regional_period_rows)
display(regional_periods.round(3))
regional_periods.to_csv(
    TABLE_DIR / "regional_g3_vza_periods.csv", index=False
)

low_day_comparison = (
    regional_angle_daily.loc[
        regional_angle_daily.RQ_comparable
    ]
    .assign(low_day=lambda d: d.matched_NTL.le(LOW_NTL))
    .groupby(["year", "low_day"])
    .agg(
        days=("date", "size"),
        median_VZA=("median_VZA", "median"),
        median_paired_SC_pct=("paired_SC_pct", "median"),
        median_NTL=("matched_NTL", "median"),
    )
    .reset_index()
)
display(low_day_comparison.round(3))
low_day_comparison.to_csv(
    TABLE_DIR / "regional_g3_low_days_vs_vza.csv", index=False
)

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=.12,
    column_widths=[.53, .47],
    subplot_titles=[
        "Observed-only candidate periods",
        "Daily regional NTL versus measured VZA",
    ],
)
for name, color in [
    ("Matched G3 NTL", "#243B5A"),
    ("Median VZA", "#0072B2"),
    ("Paired SC", "#008A3A"),
]:
    fig.add_trace(go.Scatter(
        x=PERIOD_GRID, y=regional_spectra[name],
        mode="lines", line=dict(color=color, width=2), name=name,
    ), row=1, col=1)
for period, color in [
    (16, "#777777"),
    (dominant_period_days, EVENT_COLOR),
]:
    fig.add_vline(
        x=period, row=1, col=1,
        line_color=color, line_dash="dash", line_width=1.2,
    )
for year, color in [(2012, "#243B5A"), (2013, "#E95A3D")]:
    part = regional_angle_daily.loc[
        regional_angle_daily.year.eq(year)
        & regional_angle_daily.RQ_comparable
    ]
    fig.add_trace(go.Scatter(
        x=part.median_VZA, y=part.matched_NTL,
        mode="markers",
        marker=dict(color=color, size=6, opacity=.55),
        name=f"{year} days",
        customdata=part[["paired_SC_pct", "paired_pixels"]],
        hovertemplate=(
            f"{year}<br>Median VZA %{{x:.1f}}°"
            "<br>Matched NTL %{y:.3f}"
            "<br>Paired SC %{customdata[0]:.1f}%"
            "<br>Paired pixels %{customdata[1]:.0f}"
            "<extra></extra>"
        ),
    ), row=1, col=2)
style_figure(fig, "Regional cycle versus viewing geometry", 1550, 640)
fig.update_xaxes(title_text="Period (days)", range=[8, 70], row=1, col=1)
fig.update_yaxes(title_text="Normalized power", row=1, col=1)
fig.update_xaxes(title_text="Median VZA (degrees)", row=1, col=2)
fig.update_yaxes(title_text="Matched NTL (nW cm⁻² sr⁻¹)", row=1, col=2)
finish_figure(fig, "08d_regional_g3_vza_frequency")


### 8c. Is the angle response visible within G3 pixels and GHSL classes?

A regional median can change because the visible class mix changes. Centre log1p radiance **within each pixel and calendar month**, then compare VZA bins separately for GHSL classes 22, 23 and 30 and for 2012 versus 2013. This keeps pixel brightness and slow monthly changes out of the displayed anomaly. Every populated bin is shown with its actual number of pixel-days and pixels.


In [ ]:
regional_log_ntl = np.log1p(regional_matched_ntl)
regional_monthly_log_anomaly = np.full_like(
    regional_log_ntl, np.nan, dtype="float32"
)
regional_monthly_vza_centered = np.full_like(
    regional_matched_vza, np.nan, dtype="float32"
)
regional_monthly_vza2_centered = np.full_like(
    regional_matched_vza, np.nan, dtype="float32"
)
month_index = selected_dates.to_period("M")
for month in month_index.unique():
    rows = np.flatnonzero(month_index == month)
    vza_month = regional_matched_vza[rows]
    regional_monthly_log_anomaly[rows] = (
        regional_log_ntl[rows]
        - median(regional_log_ntl[rows], axis=0)
    )
    regional_monthly_vza_centered[rows] = (
        vza_month - median(vza_month, axis=0)
    )
    regional_monthly_vza2_centered[rows] = (
        vza_month**2 - median(vza_month**2, axis=0)
    )

class_bin_rows = []
for class_code in (22, 23, 30):
    columns = np.flatnonzero(regional_classes == class_code)
    for year in (2012, 2013):
        rows = np.flatnonzero(selected_dates.year == year)
        angles = regional_matched_vza[np.ix_(rows, columns)]
        anomalies = regional_monthly_log_anomaly[
            np.ix_(rows, columns)
        ]
        for lower in range(0, 90, 10):
            upper = 90.001 if lower == 80 else lower + 10
            in_bin = (
                (angles >= lower) & (angles < upper)
                & np.isfinite(anomalies)
            )
            values = anomalies[in_bin]
            class_bin_rows.append(dict(
                GHSL_class=class_code,
                year=year,
                VZA_midpoint=lower + 5,
                median_log_anomaly=(
                    float(np.median(values)) if values.size else np.nan
                ),
                pairs=int(in_bin.sum()),
                pixels=int(in_bin.any(axis=0).sum()),
            ))
class_vza_bins = pd.DataFrame(class_bin_rows)
display(class_vza_bins.round(3))
class_vza_bins.to_csv(
    TABLE_DIR / "regional_g3_vza_by_ghsl_year.csv", index=False
)

fig = make_subplots(
    rows=1, cols=3, horizontal_spacing=.08,
    subplot_titles=[
        f"{code} · {GHSL_CLASS_LABELS[code]}"
        for code in (22, 23, 30)
    ],
)
for col, code in enumerate((22, 23, 30), start=1):
    for year, dash in [(2012, "solid"), (2013, "dash")]:
        part = class_vza_bins.loc[
            class_vza_bins.GHSL_class.eq(code)
            & class_vza_bins.year.eq(year)
        ]
        fig.add_trace(go.Scatter(
            x=part.VZA_midpoint,
            y=part.median_log_anomaly,
            mode="lines+markers", connectgaps=False,
            line=dict(
                color=GHSL_CLASS_COLORS[code],
                width=2, dash=dash,
            ),
            marker=dict(size=6),
            name=str(year), legendgroup=str(year),
            showlegend=(col == 1),
            customdata=part[["pairs", "pixels"]],
            hovertemplate=(
                f"GHSL {code} · {year}"
                "<br>VZA %{x:.0f}°"
                "<br>Within-pixel-month log anomaly %{y:.3f}"
                "<br>Pairs %{customdata[0]:.0f}"
                "<br>Pixels %{customdata[1]:.0f}"
                "<extra></extra>"
            ),
        ), row=1, col=col)
    fig.add_hline(
        y=0, line_color=BASELINE_COLOR, line_width=1,
        row=1, col=col,
    )
    fig.update_xaxes(
        title_text="Sensor zenith (degrees)",
        range=[0, 90], dtick=20, row=1, col=col,
    )
    fig.update_yaxes(
        title_text="log1p NTL − pixel-month median",
        row=1, col=col,
    )
style_figure(fig, "G3 within-pixel angle response", 1600, 620)
finish_figure(fig, "08e_regional_g3_vza_by_class")


### 8d. Would an angle adjustment improve held-out regional data?

Fit a separate quadratic angle response for each G3 GHSL class using **2012 only**, after pixel-month centring. Reference each trained pixel to its own 2012 median VZA. Apply the fixed class coefficients to 2013, then compare observed and adjusted daily medians on the **identical trained-and-observed pixel set**. This is a sensitivity test, not an adopted correction. Evaluate only 2013 dates/blocks meeting the notebook’s existing 10% SC threshold; plot all matched values and retain the support table. Report these checks for both daily values and pixel-then-spatial four-day composites: paired support, 16-day and detected-period amplitudes, MAD, near-zero periods and negative adjusted values. A smoother line alone is insufficient.


In [ ]:
class_angle_models = []
angle_coefficients_2012 = {}
for class_code in (22, 23, 30):
    columns = np.flatnonzero(regional_classes == class_code)
    for year in (2012, 2013):
        rows = np.flatnonzero(selected_dates.year == year)
        x1 = regional_monthly_vza_centered[
            np.ix_(rows, columns)
        ]
        x2 = regional_monthly_vza2_centered[
            np.ix_(rows, columns)
        ]
        y = regional_monthly_log_anomaly[
            np.ix_(rows, columns)
        ]
        valid = np.isfinite(x1) & np.isfinite(x2) & np.isfinite(y)
        if valid.sum() < 3:
            continue
        design = np.column_stack([
            np.ones(valid.sum()), x1[valid], x2[valid]
        ])
        if np.linalg.matrix_rank(design) < 3:
            continue
        beta = np.linalg.lstsq(
            design, y[valid], rcond=None
        )[0]
        class_angle_models.append(dict(
            GHSL_class=class_code,
            year=year,
            pairs=int(valid.sum()),
            pixels=int(valid.any(axis=0).sum()),
            paired_share_pct=float(100 * valid.mean()),
            linear_per_degree=float(beta[1]),
            quadratic_per_degree2=float(beta[2]),
        ))
        if year == 2012:
            angle_coefficients_2012[class_code] = beta

angle_model_table = pd.DataFrame(class_angle_models)
if not angle_coefficients_2012:
    raise ValueError("No 2012 GHSL-class VZA response is estimable.")
display(angle_model_table.round(6))
angle_model_table.to_csv(
    TABLE_DIR / "regional_g3_vza_class_models.csv", index=False
)

vza_reference_2012 = median(
    regional_matched_vza[selected_dates.year == 2012], axis=0
)
regional_corrected_log = np.full_like(
    regional_log_ntl, np.nan, dtype="float32"
)
for class_code, beta in angle_coefficients_2012.items():
    columns = np.flatnonzero(
        (regional_classes == class_code)
        & np.isfinite(vza_reference_2012)
    )
    if not len(columns):
        continue
    angle_now = regional_matched_vza[:, columns]
    angle_ref = vza_reference_2012[columns][None, :]
    modeled_angle_effect = (
        beta[1] * (angle_now - angle_ref)
        + beta[2] * (angle_now**2 - angle_ref**2)
    )
    regional_corrected_log[:, columns] = (
        regional_log_ntl[:, columns] - modeled_angle_effect
    )

regional_model_pair = np.isfinite(regional_corrected_log)
regional_model_observed = median(
    np.where(regional_model_pair, regional_matched_ntl, np.nan),
    axis=1,
)
regional_model_adjusted = median(
    np.expm1(regional_corrected_log), axis=1
)
regional_model_sc = (
    100 * regional_model_pair.sum(axis=1) / len(regional_pixels)
)
regional_model_daily = pd.DataFrame({
    "date": selected_dates,
    "year": selected_dates.year,
    "observed": regional_model_observed,
    "angle_adjusted": regional_model_adjusted,
    "paired_SC_pct": regional_model_sc,
    "paired_pixels": regional_model_pair.sum(axis=1),
    "median_VZA": median(regional_matched_vza, axis=1),
})
regional_model_daily.to_csv(
    TABLE_DIR / "regional_g3_vza_adjustment_diagnostic.csv",
    index=False,
)

# The four-day comparison preserves pixel-then-spatial compositing
# and uses exactly the same modeled pixels before and after adjustment.
regional_model_4d_rows = []
for block in baseline_4d.itertuples():
    block_id = int((block.date - EVENT_DATE).days // AGGREGATION_DAYS)
    day_rows = np.flatnonzero(regional_block_id == block_id)
    if not len(day_rows):
        continue
    pixel_observed_4d = median(
        np.where(
            regional_model_pair[day_rows],
            regional_matched_ntl[day_rows], np.nan,
        ), axis=0,
    )
    pixel_adjusted_4d = median(
        np.expm1(regional_corrected_log[day_rows]), axis=0
    )
    matched_4d = (
        np.isfinite(pixel_observed_4d)
        & np.isfinite(pixel_adjusted_4d)
    )
    regional_model_4d_rows.append(dict(
        date=block.date,
        year=block.date.year,
        RQ_NTL_4D=block.raw_median_ntl,
        observed=median(pixel_observed_4d[matched_4d]),
        angle_adjusted=median(pixel_adjusted_4d[matched_4d]),
        paired_pixels=int(matched_4d.sum()),
        paired_SC_pct=100 * matched_4d.mean(),
    ))
regional_model_4d = pd.DataFrame(regional_model_4d_rows)
regional_model_4d["RQ_comparable"] = (
    regional_model_4d.RQ_NTL_4D.notna()
    & regional_model_4d.paired_SC_pct.ge(SPATIAL_COMPLETENESS_PCT)
)
regional_model_4d.to_csv(
    TABLE_DIR / "regional_g3_vza_adjustment_4d.csv", index=False
)

def amplitude_if_estimable(series, period):
    series = series.dropna()
    if len(series) < 4:
        return np.nan
    t = (series.index - daily_calendar[0]).days.to_numpy(float)
    phase = 2 * np.pi * t / period
    design = np.column_stack([
        np.ones(len(t)), t, np.sin(phase), np.cos(phase)
    ])
    if np.linalg.matrix_rank(design) < 4:
        return np.nan
    return harmonic_amplitude(series, period)

test = regional_model_daily.loc[
    regional_model_daily.year.eq(2013)
    & regional_model_daily.paired_SC_pct.ge(SPATIAL_COMPLETENESS_PCT)
    & regional_model_daily.observed.notna()
    & regional_model_daily.angle_adjusted.notna()
].set_index("date")
test_corrected_pixels = regional_corrected_log[
    selected_dates.year == 2013
]
corrected_negative_pixel_pct = float(
    100 * ((np.expm1(test_corrected_pixels) < 0)
           & np.isfinite(test_corrected_pixels)).sum()
    / max(1, np.isfinite(test_corrected_pixels).sum())
)
regional_angle_validation_rows = []
for column, label in [
    ("observed", "Observed on modeled pixels"),
    ("angle_adjusted", "2012-angle-adjusted diagnostic"),
]:
    values = test[column]
    regional_angle_validation_rows.append(dict(
        state=label,
        test_days=len(values),
        modelled_fixed_pixels=int(
            regional_model_pair.any(axis=0).sum()
        ),
        median_paired_SC_pct=float(test.paired_SC_pct.median()),
        amplitude_16d=amplitude_if_estimable(values, 16.0),
        amplitude_detected_period=amplitude_if_estimable(
            values, dominant_period_days
        ),
        temporal_MAD=float((values - values.median()).abs().median()),
        near_zero_days=int(values.le(LOW_NTL).sum()),
        negative_daily_pct=float(100 * values.lt(0).mean()),
        negative_pixel_pct=(
            0.0 if column == "observed"
            else corrected_negative_pixel_pct
        ),
        correlation_with_median_VZA=values.corr(test.median_VZA),
    ))
regional_angle_validation = pd.DataFrame(
    regional_angle_validation_rows
)
display(regional_angle_validation.round(3))
regional_angle_validation.to_csv(
    TABLE_DIR / "regional_g3_vza_heldout_test.csv", index=False
)

heldout_4d = regional_model_4d.loc[
    regional_model_4d.year.eq(2013)
    & regional_model_4d.RQ_comparable
    & regional_model_4d.observed.notna()
    & regional_model_4d.angle_adjusted.notna()
].set_index("date")
four_day_rows = []
for column, label in [
    ("observed", "Observed on modeled pixels"),
    ("angle_adjusted", "2012-angle-adjusted diagnostic"),
]:
    values = heldout_4d[column]
    four_day_rows.append(dict(
        state=label,
        heldout_blocks=len(values),
        median_paired_SC_pct=float(heldout_4d.paired_SC_pct.median()),
        amplitude_16d=amplitude_if_estimable(values, 16.0),
        amplitude_detected_period=amplitude_if_estimable(
            values, dominant_period_days
        ),
        temporal_MAD=float((values - values.median()).abs().median()),
        near_zero_blocks=int(values.le(LOW_NTL).sum()),
        negative_blocks=int(values.lt(0).sum()),
    ))
regional_angle_validation_4d = pd.DataFrame(four_day_rows)
display(regional_angle_validation_4d.round(3))
regional_angle_validation_4d.to_csv(
    TABLE_DIR / "regional_g3_vza_heldout_test_4d.csv", index=False
)

model_plot = regional_model_daily.set_index("date").reindex(
    daily_calendar
)
adjusted_2013 = model_plot.angle_adjusted.where(
    model_plot.year.eq(2013)
)
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    row_heights=[.13, .55, .32], vertical_spacing=.07,
    subplot_titles=[
        "SC of the identical modeled pixel set",
        "Observed versus 2012-trained angle diagnostic",
        "Adjustment applied to the regional median",
    ],
)
fig.add_trace(go.Heatmap(
    x=daily_calendar, y=["Modeled G3 SC"],
    z=[model_plot.paired_SC_pct.to_numpy()],
    colorscale=SC_GREEN_COLORSCALE, zmin=0, zmax=100,
    colorbar=dict(
        title=dict(text="SC (%)", font=dict(size=16)),
        tickfont=dict(size=15), thickness=18,
        x=1.01, y=.93, len=.14,
    ),
    hovertemplate=(
        "%{x|%d %b %Y}<br>Paired SC %{z:.1f}%"
        "<extra></extra>"
    ),
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=daily_calendar, y=model_plot.observed,
    mode="lines+markers", connectgaps=False,
    line=dict(color="#243B5A", width=1.4),
    marker=dict(color="#243B5A", size=3),
    name="Observed on modeled pixels",
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=daily_calendar, y=adjusted_2013,
    mode="lines+markers", connectgaps=False,
    line=dict(color="#D55E00", width=1.7),
    marker=dict(color="#D55E00", size=3),
    name="2012-angle-adjusted · 2013 only",
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=daily_calendar,
    y=adjusted_2013 - model_plot.observed,
    mode="lines", connectgaps=False,
    line=dict(color="#D55E00", width=1.5),
    name="Adjusted − observed",
), row=3, col=1)
fig.add_hline(
    y=0, line_color=BASELINE_COLOR, line_width=1,
    row=3, col=1,
)
style_figure(fig, "Regional G3 · held-out angle sensitivity", 1550, 900)
fig.update_yaxes(showticklabels=False, row=1, col=1)
fig.update_yaxes(title_text="NTL (nW cm⁻² sr⁻¹)", row=2, col=1)
fig.update_yaxes(title_text="NTL difference", row=3, col=1)
fig.update_xaxes(
    range=[CHARACTERIZATION_START, PRE_EVENT_END],
    dtick="M3", tickformat="%b %Y", row=3, col=1,
)
finish_figure(fig, "08f_regional_g3_vza_heldout")

fig = go.Figure()
for column, label, color, dash in [
    ("observed", "Observed matched NTL (4D)", "#243B5A", "solid"),
    ("angle_adjusted", "Angle-adjusted diagnostic (4D)", "#D55E00", "dash"),
]:
    values = regional_model_4d[column].where(
        regional_model_4d.year.eq(2013)
    )
    fig.add_trace(go.Scatter(
        x=regional_model_4d.date, y=values,
        mode="lines+markers", connectgaps=False,
        line=dict(color=color, width=2, dash=dash),
        marker=dict(size=4), name=label,
        customdata=regional_model_4d[["paired_pixels", "paired_SC_pct"]],
        hovertemplate=(
            "%{x|%d %b %Y}<br>NTL %{y:.3f}"
            "<br>Paired pixels %{customdata[0]:.0f}"
            "<br>Paired SC %{customdata[1]:.1f}%"
            "<extra></extra>"
        ),
    ))
style_figure(fig, "Regional G3 · held-out four-day comparison", 1450, 600)
fig.update_xaxes(dtick="M2", tickformat="%b %Y")
fig.update_yaxes(title_text="NTL (nW cm⁻² sr⁻¹)")
finish_figure(fig, "08f_regional_g3_vza_heldout_4d")


**Decision at regional scale.** An angle adjustment becomes a credible *sensitivity analysis* only if (1) low NTL days coincide with distinctive measured angles on matched pixels, (2) the relationship repeats within G3 classes and across years, and (3) the 2012-trained response reduces the 2013 cycle without relying on lower support or creating implausible negative values. If these fail, preserve the unadjusted reliability-qualified series and carry the cyclicity as unresolved baseline uncertainty. The selected bright/dark controls below check whether the regional result is widespread or driven by particular pixels.


### 8e. Do selected bright and dark controls reproduce the regional timing?

Use only same-date, same-pixel VZA/MQF=0 NTL pairs. The green strip is **paired** spatial completeness (valid pairs divided by fixed pixels), not an additional selection gate. Both years appear on the same time axis, and the table reports each year's paired days and candidate VZA period. Missing dates remain visible as gaps.


In [ ]:
if paired_angle.empty:
    print("Measured-VZA temporal comparison skipped: no paired angle data.")
else:
    daily_angle = paired_angle.groupby(["date", "group"]).agg(
        median_VZA=("sensor_zenith_deg", "median"),
        median_NTL=("ntl", "median"),
        paired_pixels=("pixel_index", "nunique"),
    ).reset_index()
    angle_wide = daily_angle.pivot(index="date", columns="group",
                                   values="median_VZA").reindex(index=daily_calendar, columns=list(pixel_groups))
    ntl_wide = daily_angle.pivot(index="date", columns="group",
                                 values="median_NTL").reindex(index=daily_calendar, columns=list(pixel_groups))
    paired_count = daily_angle.pivot(index="date", columns="group",
                                     values="paired_pixels").reindex(index=daily_calendar, columns=list(pixel_groups)).fillna(0)
    paired_sc = pd.DataFrame({
        name: 100 * paired_count[name] / len(pixel_groups[name])
        for name in pixel_groups
    }, index=daily_calendar)
    vza_peaks = []
    for name in pixel_groups:
        if name not in angle_wide:
            print(f"{name}: no paired VZA/NTL dates.")
            continue
        for year in (2012, 2013):
            part = angle_wide[name].loc[angle_wide.index.year == year]
            spectrum = observed_spectrum(part, part.index)
            vza_peaks.append({
                "group": name, "year": year,
                "measured_VZA_peak_8_to_60_days": strongest(PERIOD_GRID, spectrum),
                "VZA_power_at_16_days": spectrum[int(abs(PERIOD_GRID - 16).argmin())],
                "paired_days": int(part.notna().sum()),
            })
    display(pd.DataFrame(vza_peaks).round(2))
    pd.DataFrame(vza_peaks).to_csv(TABLE_DIR / "measured_vza_periods.csv",
                                    index=False)

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
        row_heights=[.15, .38, .47], vertical_spacing=.06,
        subplot_titles=["Paired fixed-pixel SC", "Measured sensor zenith angle",
                        "NTL on the same paired pixel-days"])
    group_order = ["Bright urban", "Dark donor", "Dark holdout"]
    fig.add_trace(go.Heatmap(
        x=daily_calendar, y=group_order,
        z=np.vstack([
            paired_sc[name].to_numpy()
            for name in group_order
        ]),
        colorscale=SC_GREEN_COLORSCALE, zmin=0, zmax=100,
        colorbar=dict(title="SC (%)", thickness=18,
                      tickfont=dict(size=15), x=1.01, y=.92, len=.15),
    ), row=1, col=1)
    for name, color in [
        ("Bright urban", "#E95A3D"),
        ("Dark donor", "#0072B2"),
        ("Dark holdout", "#56B4E9"),
    ]:
        if name not in angle_wide:
            continue
        fig.add_trace(go.Scatter(
            x=daily_calendar, y=angle_wide[name], mode="lines+markers",
            connectgaps=False, line=dict(color=color, width=1),
            marker=dict(color=color, size=4, opacity=.7),
            name=name + " VZA",
        ), row=2, col=1)
        fig.add_trace(go.Scatter(
            x=daily_calendar, y=ntl_wide[name], mode="lines+markers",
            connectgaps=False, line=dict(color=color, width=1),
            marker=dict(color=color, size=4, opacity=.7),
            name=name + " NTL",
        ), row=3, col=1)
    style_figure(fig, "Measured angle and matched DNB-BRDF", 1550, 920)
    fig.update_yaxes(showticklabels=False, row=1, col=1)
    fig.update_yaxes(title_text="VZA (degrees)", row=2, col=1)
    fig.update_yaxes(title_text="NTL (nW cm⁻² sr⁻¹)", row=3, col=1)
    fig.update_xaxes(range=[CHARACTERIZATION_START, PRE_EVENT_END],
                     dtick="M3", tickformat="%b %Y", row=3, col=1)
    finish_figure(fig, "08c_measured_vza_timeline")


### 8f. Does the control-pixel response recur within pixels?

Centre NTL within each pixel, then compare 10° VZA bins by group and year. **All observed bins are displayed**; the table and hover report pair and pixel counts so sparsely supported bins can be judged rather than silently removed. Replication between years matters more than one pooled slope.


In [ ]:
if paired_angle.empty:
    print("Within-pixel VZA comparison skipped: no paired angle data.")
else:
    paired_angle["VZA_bin"] = pd.cut(
        paired_angle.sensor_zenith_deg,
        bins=[0, 10, 20, 30, 40, 50, 60, 70, 80, 90.001], right=False,
    )
    vza_bins = paired_angle.groupby(
        ["group", "year", "VZA_bin"], observed=True
    ).agg(
        median_anomaly=("ntl_within_pixel", "median"),
        pairs=("ntl", "size"),
        pixels=("pixel_index", "nunique"),
    ).reset_index()
    vza_bins["VZA_midpoint"] = vza_bins.VZA_bin.map(
        lambda interval: interval.mid
    ).astype(float)
    display(vza_bins.round(2))
    vza_bins.to_csv(TABLE_DIR / "vza_within_pixel_by_year.csv", index=False)

    fig = make_subplots(rows=1, cols=3, horizontal_spacing=.09,
        subplot_titles=["Bright urban", "Dark donor", "Dark holdout"])
    for col, (name, color) in enumerate([
        ("Bright urban", "#E95A3D"),
        ("Dark donor", "#0072B2"),
        ("Dark holdout", "#56B4E9"),
    ], start=1):
        for year, dash in [(2012, "solid"), (2013, "dash")]:
            part = vza_bins.loc[
                vza_bins.group.eq(name) & vza_bins.year.eq(year)
            ].sort_values("VZA_midpoint")
            fig.add_trace(go.Scatter(
                x=part.VZA_midpoint, y=part.median_anomaly,
                mode="lines+markers", connectgaps=False,
                line=dict(color=color, width=2, dash=dash),
                marker=dict(size=6), name=str(year),
                legendgroup=str(year), showlegend=(col == 1),
                customdata=part[["pairs", "pixels"]],
                hovertemplate=(
                    f"{name} · {year}<br>VZA %{{x:.0f}}°"
                    "<br>Within-pixel NTL anomaly %{y:.2f}"
                    "<br>Pairs %{customdata[0]:.0f}"
                    "<br>Pixels %{customdata[1]:.0f}<extra></extra>"
                ),
            ), row=1, col=col)
        fig.add_hline(y=0, line_color=BASELINE_COLOR,
                      line_width=1, row=1, col=col)
        fig.update_xaxes(title_text="VZA (degrees)", range=[0, 90],
                         dtick=20, row=1, col=col)
        fig.update_yaxes(title_text="NTL − pixel median", row=1, col=col)
    style_figure(fig, "Measured VZA and within-pixel radiance", 1600, 610)
    finish_figure(fig, "08d_vza_within_pixel_by_year")


### 8g. Does viewing direction matter for the controls?

If A1 contains `Sensor_Azimuth`, compare two look-direction halves by group and year. Angle magnitude alone can miss directional lighting. Every observed bin is retained and its support is reported. If azimuth is absent, this remains a documented limit rather than a guessed variable.


In [ ]:
if (paired_angle.empty or "sensor_azimuth_deg" not in paired_angle.columns
        or paired_angle.sensor_azimuth_deg.notna().sum() == 0):
    print("Viewing-direction check skipped: paired Sensor_Azimuth not available.")
else:
    view = paired_angle.dropna(subset=["sensor_azimuth_deg"]).copy()
    view["azimuth_half"] = pd.cut(
        np.mod(view.sensor_azimuth_deg, 360),
        bins=[0, 180, 360], labels=["0–180°", "180–360°"],
        right=False, include_lowest=True,
    )
    az_bins = view.groupby(
        ["group", "year", "azimuth_half", "VZA_bin"],
        observed=True,
    ).agg(
        median_anomaly=("ntl_within_pixel", "median"),
        pairs=("ntl", "size"),
        pixels=("pixel_index", "nunique"),
    ).reset_index()
    az_bins["VZA_midpoint"] = az_bins.VZA_bin.map(
        lambda interval: interval.mid
    ).astype(float)
    display(az_bins.round(2))
    az_bins.to_csv(TABLE_DIR / "vza_by_view_direction.csv", index=False)
    fig = make_subplots(rows=2, cols=2, shared_xaxes=True,
        horizontal_spacing=.12, vertical_spacing=.12,
        subplot_titles=[
            "Bright urban · 2012", "Dark donor · 2012",
            "Bright urban · 2013", "Dark donor · 2013",
        ])
    for row, year in enumerate((2012, 2013), start=1):
        for col, group in enumerate(("Bright urban", "Dark donor"), start=1):
            for half, color in [
                ("0–180°", "#0072B2"), ("180–360°", "#D55E00"),
            ]:
                part = az_bins.loc[
                    az_bins.group.eq(group) & az_bins.year.eq(year)
                    & az_bins.azimuth_half.eq(half)
                ].sort_values("VZA_midpoint")
                fig.add_trace(go.Scatter(
                    x=part.VZA_midpoint, y=part.median_anomaly,
                    mode="lines+markers", connectgaps=False,
                    line=dict(color=color, width=2),
                    marker=dict(size=5), name=half,
                    legendgroup=half, showlegend=(row == 1 and col == 1),
                    customdata=part[["pairs", "pixels"]],
                    hovertemplate=(
                        f"{group} · {year} · {half}"
                        "<br>VZA %{x:.0f}°"
                        "<br>Within-pixel NTL anomaly %{y:.2f}"
                        "<br>Pairs %{customdata[0]:.0f}"
                        "<br>Pixels %{customdata[1]:.0f}<extra></extra>"
                    ),
                ), row=row, col=col)
            fig.add_hline(y=0, line_color=BASELINE_COLOR,
                          line_width=1, row=row, col=col)
            fig.update_yaxes(title_text="NTL − pixel median",
                             row=row, col=col)
            fig.update_xaxes(range=[0, 90], dtick=20,
                             title_text="VZA (degrees)" if row == 2 else None,
                             row=row, col=col)
    style_figure(fig, "Angular magnitude and look direction", 1500, 790)
    finish_figure(fig, "08e_vza_by_view_direction")


### 8h. Does the control contrast survive pixel-month matching?

Compare high- and low-angle radiance within the **same pixel and calendar month**, using quartiles of the observed VZA distribution. A pixel-month contributes only if both angle bands were observed; this is a pairing requirement, not a percentage gate. Report results separately for 2012 and 2013, including the number of matched pixel-months.


In [ ]:
if paired_angle.empty:
    print("Within-month VZA contrast skipped: no paired angle data.")
else:
    low_angle, high_angle = paired_angle.sensor_zenith_deg.quantile([.25, .75])
    angle_band = np.select(
        [paired_angle.sensor_zenith_deg.le(low_angle),
         paired_angle.sensor_zenith_deg.ge(high_angle)],
        ["Low VZA", "High VZA"], default="Middle",
    )
    angle_extremes = paired_angle.assign(angle_band=angle_band)
    angle_extremes = angle_extremes.loc[
        angle_extremes.angle_band.ne("Middle")
    ]
    paired_month = angle_extremes.groupby(
        ["group", "pixel_index", "month", "angle_band"]
    ).ntl.median().unstack("angle_band")
    paired_month = paired_month.dropna(
        subset=["Low VZA", "High VZA"]
    ).copy()
    paired_month["high_minus_low"] = (
        paired_month["High VZA"] - paired_month["Low VZA"]
    )
    print(f"Observed VZA quartiles: ≤{low_angle:.1f}° and ≥{high_angle:.1f}°")
    if paired_month.empty:
        print("No pixel-month contains both angle bands; contrast unresolved.")
    else:
        paired_month = paired_month.reset_index()
        paired_month["year"] = paired_month.month.dt.year
        contrast_table = paired_month.groupby(["year", "group"]).agg(
            pixel_months=("high_minus_low", "size"),
            pixels=("pixel_index", "nunique"),
            median_high_minus_low=("high_minus_low", "median"),
            q25=("high_minus_low", lambda x: x.quantile(.25)),
            q75=("high_minus_low", lambda x: x.quantile(.75)),
        )
        display(contrast_table.round(3))
        paired_month.to_csv(
            TABLE_DIR / "vza_pixel_month_contrasts.csv", index=False
        )
        fig = make_subplots(
            rows=1, cols=2, horizontal_spacing=.12,
            subplot_titles=["2012", "2013"],
        )
        for col, year in enumerate((2012, 2013), start=1):
            for name, color in [
                ("Bright urban", "#E95A3D"),
                ("Dark donor", "#0072B2"),
                ("Dark holdout", "#56B4E9"),
            ]:
                part = paired_month.loc[
                    paired_month.year.eq(year) & paired_month.group.eq(name)
                ]
                fig.add_trace(go.Box(
                    y=part.high_minus_low, name=name,
                    legendgroup=name, showlegend=(col == 1),
                    marker=dict(color=color, size=5),
                    line=dict(color=color, width=1.5),
                    boxpoints="all", jitter=.35, pointpos=0,
                    fillcolor="rgba(0,0,0,0)",
                ), row=1, col=col)
            fig.add_hline(
                y=0, line_color=BASELINE_COLOR, line_width=1,
                row=1, col=col,
            )
            fig.update_yaxes(
                title_text="High − low VZA NTL (nW cm⁻² sr⁻¹)",
                row=1, col=col,
            )
        style_figure(fig, "High minus low VZA within pixel-month", 1400, 630)
        finish_figure(fig, "08f_vza_pixel_month_contrast")


### 8i. Does a control-pixel angle relationship transfer to 2013?

Fit angle response on 2012 bright pixels and, separately, 2012 dark-donor pixels. Apply the fixed responses to 2013 bright and dark-holdout pixels. The plot includes the **2012 training record** and the **2013 observed and adjusted diagnostic** over the full pre-event axis. Every matched day is retained, with paired SC shown above; the 16-day amplitude and temporal MAD are reported as tests, not as permission to correct the primary NTL series.


In [ ]:
vza_validation_results = pd.DataFrame()
reference_2012 = (
    paired_angle.loc[paired_angle.year.eq(2012)]
    .groupby("pixel_index").ntl.median()
)
model_rows = paired_angle.join(
    reference_2012.rename("pixel_NTL_2012"),
    on="pixel_index",
).dropna(subset=["pixel_NTL_2012"]).copy()
model_rows["anomaly_2012_ref"] = (
    model_rows.ntl - model_rows.pixel_NTL_2012
)

validation_rows = []
plotted_series = {}
for train_group, test_group in [
    ("Bright urban", "Bright urban"),
    ("Dark donor", "Dark holdout"),
]:
    train = model_rows.loc[
        model_rows.year.eq(2012) & model_rows.group.eq(train_group)
    ].copy()
    test = model_rows.loc[
        model_rows.year.eq(2013) & model_rows.group.eq(test_group)
    ].copy()
    if train.empty or test.empty:
        print(f"{test_group}: no matched 2012 training or 2013 test pairs.")
        continue

    centre_angle = float(train.sensor_zenith_deg.median())
    train_z = train.sensor_zenith_deg.to_numpy() - centre_angle
    design = np.column_stack([
        np.ones(len(train_z)), train_z, train_z**2,
    ])
    if np.linalg.matrix_rank(design) < 3:
        print(f"{train_group}: angle model is rank-deficient.")
        continue
    beta = np.linalg.lstsq(
        design, train.anomaly_2012_ref.to_numpy(), rcond=None
    )[0]

    test_z = test.sensor_zenith_deg.to_numpy() - centre_angle
    test["angle_adjusted"] = (
        test.anomaly_2012_ref
        - (beta[1] * test_z + beta[2] * test_z**2)
    )
    daily_train = train.groupby("date").agg(
        observed=("anomaly_2012_ref", "median"),
        paired_pixels=("pixel_index", "nunique"),
    )
    daily_test = test.groupby("date").agg(
        observed=("anomaly_2012_ref", "median"),
        adjusted=("angle_adjusted", "median"),
        paired_pixels=("pixel_index", "nunique"),
    )

    # Four dates are the mathematical minimum for the four-parameter
    # intercept + trend + sine + cosine diagnostic, not an SC gate.
    if len(daily_test) >= 4:
        before = harmonic_amplitude(daily_test.observed, 16.0)
        after = harmonic_amplitude(daily_test.adjusted, 16.0)
    else:
        before, after = np.nan, np.nan

    validation_rows.append(dict(
        train_group=train_group,
        test_group=test_group,
        validation_year=2013,
        training_days=len(daily_train),
        paired_days=len(daily_test),
        median_paired_pixels=daily_test.paired_pixels.median(),
        amplitude_16d_before=before,
        amplitude_16d_after=after,
        MAD_before=(
            daily_test.observed - daily_test.observed.median()
        ).abs().median(),
        MAD_after=(
            daily_test.adjusted - daily_test.adjusted.median()
        ).abs().median(),
    ))
    plotted_series[test_group] = (daily_train, daily_test, train_group)

vza_validation_results = pd.DataFrame(validation_rows)
display(vza_validation_results.round(3))
if not vza_validation_results.empty:
    vza_validation_results.to_csv(
        TABLE_DIR / "vza_2012_to_2013_test.csv", index=False
    )

if plotted_series:
    validation_calendar = daily_calendar
    groups = ["Bright urban", "Dark donor", "Dark holdout"]
    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True,
        row_heights=[.16, .42, .42], vertical_spacing=.07,
        subplot_titles=[
            "Paired fixed-pixel SC",
            "Bright urban · 2012 fit, 2013 test",
            "Dark donor · 2012 fit; dark holdout · 2013 test",
        ],
    )
    fig.add_trace(go.Heatmap(
        x=validation_calendar, y=groups,
        z=np.vstack([
            paired_sc[name].reindex(validation_calendar).to_numpy()
            for name in groups
        ]),
        colorscale=SC_GREEN_COLORSCALE, zmin=0, zmax=100,
        colorbar=dict(
            title=dict(text="SC (%)", font=dict(size=16)),
            thickness=18, tickfont=dict(size=15),
            x=1.01, y=.92, len=.16,
        ),
        hovertemplate=(
            "%{y}<br>%{x|%d %b %Y}"
            "<br>Paired SC %{z:.1f}%<extra></extra>"
        ),
    ), row=1, col=1)

    for row, name, color in [
        (2, "Bright urban", "#E95A3D"),
        (3, "Dark holdout", "#0072B2"),
    ]:
        if name not in plotted_series:
            continue
        daily_train, daily_test, train_group = plotted_series[name]
        for values, trace_name, trace_color, dash in [
            (
                daily_train.observed,
                f"{train_group} · 2012 training",
                "#929BA5", "solid",
            ),
            (
                daily_test.observed,
                f"{name} · 2013 observed",
                color, "solid",
            ),
            (
                daily_test.adjusted,
                f"{name} · angle-adjusted diagnostic",
                "#767D88", "dash",
            ),
        ]:
            fig.add_trace(go.Scatter(
                x=validation_calendar,
                y=values.reindex(validation_calendar),
                mode="lines+markers" if dash == "solid" else "lines",
                connectgaps=False,
                line=dict(color=trace_color, width=1.5, dash=dash),
                marker=dict(color=trace_color, size=3),
                name=trace_name,
            ), row=row, col=1)
        fig.update_yaxes(title_text="NTL anomaly", row=row, col=1)

    style_figure(fig, "2012-trained angular response · 2013 test", 1550, 850)
    fig.update_yaxes(showticklabels=False, row=1, col=1)
    fig.update_xaxes(
        range=[CHARACTERIZATION_START, PRE_EVENT_END],
        dtick="M3", tickformat="%b %Y", row=3, col=1,
    )
    finish_figure(fig, "08g_vza_heldout_2013")


## 9. Test a dark-pixel adjustment on independent 2013 pixels

Only after source, sampling and angle checks, compare bright and held-out dark pixels with and without subtracting the 2012-referenced dark-donor anomaly. Use dates with actual MQF=0 observations in all three groups; retain and display each group's SC instead of imposing the former 50% rule. Report the detected-period amplitude, temporal MAD and negative values. This remains a diagnostic: it never replaces reliability-qualified NTL.


In [ ]:
donor = group_daily["Dark donor NTL"]
holdout = group_daily["Dark holdout NTL"]
bright = group_daily["Bright urban NTL"]
donor_reference_2012 = donor.loc[donor.index.year == 2012].median()
donor_anomaly = donor - donor_reference_2012
validation = pd.DataFrame({
    "dark_donor": donor,
    "dark_holdout": holdout,
    "bright": bright,
    "dark_anomaly": donor_anomaly,
}).loc["2013-01-01":PRE_EVENT_END].dropna()
normalization_table = pd.DataFrame()
if len(validation) < 30:
    print(f"Only {len(validation)} matched 2013 days; dark-control test remains inconclusive.")
else:
    validation["Bright urban SC"] = group_daily.loc[validation.index, "Bright urban SC"]
    validation["Dark donor SC"] = group_daily.loc[validation.index, "Dark donor SC"]
    validation["Dark holdout SC"] = group_daily.loc[validation.index, "Dark holdout SC"]
    validation["dark_holdout_adjusted"] = validation.dark_holdout - validation.dark_anomaly
    validation["bright_adjusted"] = validation.bright - validation.dark_anomaly

    normalization_rows = []
    for name in ("dark_holdout", "bright"):
        for state, values in [
            ("Observed", validation[name]),
            ("After donor subtraction", validation[f"{name}_adjusted"]),
        ]:
            normalization_rows.append(dict(
                group=name, state=state, matched_days=len(values),
                harmonic_amplitude=harmonic_amplitude(values, dominant_period_days),
                temporal_MAD=float((values-values.median()).abs().median()),
                median_NTL=float(values.median()),
                negative_share_pct=100*float(values.lt(0).mean()),
                median_group_SC_pct=float(validation[
                    "Dark holdout SC" if name == "dark_holdout" else "Bright urban SC"
                ].median()),
            ))
    normalization_table = pd.DataFrame(normalization_rows)
    display(normalization_table.round(3))
    normalization_table.to_csv(TABLE_DIR / "dark_control_heldout_test.csv", index=False)
    print("Donor 2012 reference median:", round(donor_reference_2012, 3))
    print("Dark-donor near-floor share on matched dates (%):",
          round(group_daily.loc[validation.index, "Dark donor floor %"].median(), 1))


    validation_calendar = pd.date_range("2013-01-01", PRE_EVENT_END, freq="D")
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
        row_heights=[.16, .42, .42], vertical_spacing=.08,
        subplot_titles=["Spatial completeness of fixed pixels",
                        "Bright urban pixels · matched 2013 dates",
                        "Held-out dark pixels · matched 2013 dates"])
    group_order = ["Bright urban", "Dark donor", "Dark holdout"]
    fig.add_trace(go.Heatmap(
        x=validation_calendar, y=group_order,
        z=np.vstack([group_daily[f"{name} SC"].reindex(validation_calendar).to_numpy()
                     for name in group_order]),
        colorscale=SC_GREEN_COLORSCALE, zmin=0, zmax=100,
        colorbar=dict(title=dict(text="SC (%)", font=dict(size=16)),
                      thickness=18, tickfont=dict(size=15),
                      x=1.01, y=.92, len=.16),
        hovertemplate="%{y}<br>%{x|%d %b %Y}<br>SC %{z:.1f}%<extra></extra>",
    ), row=1, col=1)
    for row, stem, color in [
        (2, "bright", "#E95A3D"), (3, "dark_holdout", "#0072B2")
    ]:
        observed = validation[stem].reindex(validation_calendar)
        adjusted = validation[f"{stem}_adjusted"].reindex(validation_calendar)
        fig.add_trace(go.Scatter(
            x=validation_calendar, y=observed, mode="lines+markers",
            line=dict(color=color, width=1.4), marker=dict(size=3.5),
            name=f"{stem.replace('_', ' ')} · observed",
            connectgaps=False), row=row, col=1)
        fig.add_trace(go.Scatter(
            x=validation_calendar, y=adjusted, mode="lines",
            line=dict(color="#767D88", width=1.8, dash="dash"),
            name=f"{stem.replace('_', ' ')} · adjusted",
            connectgaps=False), row=row, col=1)
        fig.update_yaxes(title_text="NTL (nW cm⁻² sr⁻¹)", row=row, col=1)
    style_figure(fig, "Held-out dark-control test", 1550, 850)
    fig.update_yaxes(title_text=None, showticklabels=False, row=1, col=1)
    fig.update_xaxes(range=[pd.Timestamp("2013-01-01"), PRE_EVENT_END],
                     dtick="M2", tickformat="%b %Y", row=3, col=1)
    finish_figure(fig, "07_dark_signal_heldout_test")


## 10. Evidence ledger: what survived each diagnostic?

Use the ledger after running every cell. It juxtaposes source/compositing evidence, sampling and frequency checks, pixel coherence, measured-angle support and held-out adjustments. A descriptive STL component remains descriptive unless these independent checks support an interpretation.

The regional VZA sensitivity table is read alongside the selected-pixel and dark-control tests; none alone licenses a correction.


In [ ]:
qualified_low = low_audit.loc[
    low_audit.SC_pct.ge(SPATIAL_COMPLETENESS_PCT)
]
reference_ratio = (
    qualified_low.reference_median_of_visible_pixels /
    qualified_low.reference_median_all_fixed.replace(0, np.nan)
)
peak_lookup = peak_table.set_index("signal")["peak_8_to_60_days"]
coh_lookup = (coherence.set_index(["group", "period_label"])
              if not coherence.empty else pd.DataFrame())
normal_lookup = (normalization_table.set_index(["group", "state"])
                 if not normalization_table.empty else pd.DataFrame())
def evidence_value(frame, index, column):
    try:
        return float(frame.loc[index, column])
    except (KeyError, TypeError):
        return np.nan
ledger_rows = [
    dict(question="Can low days pass the SC gate?",
         evidence=f"{len(qualified_low)} / {len(low_audit)} low days",
         implication="A qualified low day still needs source and composition checks."),
    dict(question="Were visible pixels already dim in NTL₀?",
         evidence=f"Median visible/all NTL₀ ratio {reference_ratio.median():.2f}",
         implication="Low ratio supports changing contributor composition."),
    dict(question="Are qualified low days concentrated at the floor?",
         evidence=f"Median share at 0.01: {qualified_low.at_0_01_pct.median():.1f}%",
         implication="High floor share weakens a physical-zero interpretation."),
    dict(question="Does gap-filled NTL disagree on low days?",
         evidence=f"Median gap-filled NTL {qualified_low.gap_median.median():.3f}",
         implication="Diagnostic comparison only; gap-filled values are not observations."),
    dict(question="Does the regional period repeat by year?",
         evidence=(f"2012 {peak_lookup['Regional 2012']:.1f} d; "
                   f"2013 {peak_lookup['Regional 2013']:.1f} d"),
         implication="A shifting peak weakens a stable STL cycle."),
    dict(question="Does the same period occur in the observation window?",
         evidence=(f"RQ {period_checks.iloc[-1].RQ_power:.2f}; "
                   f"SC {period_checks.iloc[-1].SC_power:.2f}; "
                   f"calendar {period_checks.iloc[-1].observation_calendar_power:.2f}"),
         implication="Shared sampling power calls for an alias/observability check."),
    dict(question="Is pixel phase coherent?",
         evidence=(f"Bright R={evidence_value(coh_lookup, ('Bright urban','All'), 'phase_concentration'):.2f}; "
                   f"dark donor R={evidence_value(coh_lookup, ('Dark donor','All'), 'phase_concentration'):.2f}"),
         implication="Compare phase concentration with floor share and both years."),
    dict(question="Does dark subtraction improve held-out pixels?",
         evidence=(f"Held-out amplitude "
                   f"{evidence_value(normal_lookup, ('dark_holdout','Observed'), 'harmonic_amplitude'):.3f} → "
                   f"{evidence_value(normal_lookup, ('dark_holdout','After donor subtraction'), 'harmonic_amplitude'):.3f}"),
         implication="Also inspect bright change, MAD and negative values."),
    dict(question="Is VZA measured on matching pixel-days?",
         evidence=(f"{len(paired_angle):,} pairs" if not paired_angle.empty
                   else "No paired VNP46A1 Sensor_Zenith yet"),
         implication="Calendar phase is not a substitute for measured VZA."),
    dict(question="Does a 2012 regional angle model improve 2013?",
         evidence=(f"16-d amplitude {regional_angle_validation.iloc[0].amplitude_16d:.3f} → "
                   f"{regional_angle_validation.iloc[1].amplitude_16d:.3f}; "
                   f"matched days {regional_angle_validation.iloc[0].test_days:.0f}"),
         implication="Read class replication, SC, MAD and negative values before considering correction."),
    dict(question="Does the four-day regional cycle survive angle adjustment?",
         evidence=(f"2013 4D 16-d amplitude "
                   f"{regional_angle_validation_4d.iloc[0].amplitude_16d:.3f} → "
                   f"{regional_angle_validation_4d.iloc[1].amplitude_16d:.3f}; "
                   f"qualified blocks {regional_angle_validation_4d.iloc[0].heldout_blocks:.0f}"),
         implication="The original RQ series stays primary; compare matched support and negative values."),
]
if not vza_validation_results.empty:
    for result in vza_validation_results.itertuples():
        ledger_rows.append(dict(
            question=f"Does a 2012 angle model transfer to {result.test_group}?",
            evidence=(f"2013 16-d amplitude "
                      f"{result.amplitude_16d_before:.3f} → "
                      f"{result.amplitude_16d_after:.3f}"),
            implication="A reduction supports, but does not prove, an angular contribution.",
        ))
ledger_table = pd.DataFrame(ledger_rows)
display(ledger_table)
ledger_table.to_csv(TABLE_DIR / "cyclicity_evidence_ledger.csv", index=False)


### Interpretation after the checks

- **Processing or observability:** low days arise from quality/compositing choices or a different mix of visible pixels, or the candidate period coincides with SC or the sampling window. Carry the apparent cycle as baseline uncertainty.
- **Viewing geometry:** the signal recurs in fixed pixels and both years; same-pixel A1 VZA has compatible timing; within-pixel and pixel-month contrasts agree; a 2012 angle response reduces the 2013 signal in independent pixels. This supports an angular contribution but does not establish a correction.
- **Other recurring influence:** a peak persists after sampling and viewing checks. Lunar, atmospheric, weather and activity hypotheses then need their own measured covariates; period coincidence alone is insufficient.
- **Unresolved:** mixed evidence leaves the STL repeating component unsuitable as an assumed physical seasonal driver or an automatic business-as-usual projection.

Keep the distinction between *unobserved*, *observed but low*, and *not yet interpretable* explicit in subsequent disaster-impact analysis.
